<a href="https://colab.research.google.com/github/BenMillerDev/Applied-LLM-Systems/blob/week-8-capstone/week-8/week8_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Integrated LLM System: NailSalonApp Owner Assistant
### COSC 650 Applied LLM Systems
### Week 8 Capstone Assignment
### Ben Miller

# Part 1: Integrate
Combine at least three techniques from earlier weeks (prompt evaluation, retrieval-augmented generation, tool use, or dynamic few-shot) into a single system that solves a real problem. Reuse your prior components rather than rebuilding them, and cite in the README which week each came from.

### Load the Gemini API key from Colab secrets

Reused from Week 4.

In [1]:
import os

try:
    from google.colab import userdata
    _key = userdata.get('GEMINI_API_KEY')
    if _key:
        os.environ['GEMINI_API_KEY'] = _key
except Exception:
    pass  # not running in Colab, or the secret isn't set there -- fall back to whatever's already in the environment

HAS_API_KEY = bool(os.environ.get('GEMINI_API_KEY', '').strip())
print('GEMINI_API_KEY set:', HAS_API_KEY)

GEMINI_API_KEY set: True


### Constants used across the notebook

In [2]:
import ast, json, math, operator, pathlib, re, shutil, signal, textwrap
from datetime import datetime, timedelta

# Models
MODEL = 'gemini-3.5-flash-lite'   # the assistant itself, plus HyDE
JUDGE_MODEL = 'gemini-3.5-flash'  # a larger model grades the answers, so the assistant isn't grading its own work
TEMPERATURE = 0.0                 # same question -> same answer, so eval scores don't drift between runs
EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'  # Week 5
RERANK_MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'   # Week 6

# Retrieval (Weeks 5 and 6)
TOP_K = 3         # chunks handed to the model per search_docs call
CANDIDATE_N = 12  # chunks FAISS pulls before the cross-encoder reranks them down to TOP_K
USE_HYDE = True   # starting values; Part 2's retrieval comparison decides the final settings
USE_RERANK = True
CHUNKING = 'paragraph'  # Week 5's paragraph chunking; Part 3 switches this to its fixed version

# Prompt (Week 3)
PROMPT_VERSION = 'v1'  # which prompts/assistant_<version>.md file the assistant runs with
JUDGE_PROMPT_VERSION = 'v1'  # which prompts/judge_<version>.md file the Part 2 judge runs with

# Agent loop (Week 4)
MAX_TURNS = 6

# Tool safety
# In the real app this would be the uid from the owner's verified Firebase Auth
# token. It's passed to the tools by dispatch(), never by the model, so a prompt
# can't point a tool at another salon's data.
OWNER_ID = 'salon-owner-1'
ALLOW_WRITE_TOOLS = False  # read-only for now: dispatch() refuses any tool not marked 'access': 'read'

# timeSlots.ts reads the clock with `new Date()`. Here "now" is fixed so the same
# question gets the same open slots on every run. 2026-10-12 is a Monday.
NOW = datetime(2026, 10, 12, 10, 0)

# Booking rules hard-coded in timeSlots.ts (not owner settings)
SLOT_INCREMENT_MINUTES = 15   # candidate start times step every 15 minutes
DEFAULT_BUFFER_MINUTES = 15   # bufferTime's default when the owner hasn't set one
MINIMUM_ADVANCE_MINUTES = 60  # same-day bookings must start more than 1 hour from now
BOOKED_STATUSES = ['pending', 'confirmed']  # appointments that still hold their time (cancelled and completed don't)

# Paths
REPO_URL = 'https://github.com/BenMillerDev/Applied-LLM-Systems.git'
BRANCH = 'week-8-capstone'
DOCS_DIR = pathlib.Path('docs')
PROMPTS_DIR = pathlib.Path('prompts')

### Download documents and prompts from my repo

The 11 NailSalonApp docs from Week 5, plus this week's versioned prompts.

In [3]:
REQUIRED_PROMPT_FILES = [
    PROMPTS_DIR / f'assistant_{PROMPT_VERSION}.md',
    PROMPTS_DIR / f'judge_{JUDGE_PROMPT_VERSION}.md',
]
missing_prompt_files = [prompt_path for prompt_path in REQUIRED_PROMPT_FILES if not prompt_path.exists()]

if not DOCS_DIR.exists() or missing_prompt_files:
    print('downloading from the repo; missing prompt files:', [str(path) for path in missing_prompt_files])
    !rm -rf _repo
    !git clone -b {BRANCH} {REPO_URL} _repo
    !cp -r _repo/week-5/docs .
    !mkdir -p {PROMPTS_DIR}
    !cp _repo/week-8/prompts/*.md {PROMPTS_DIR}/

DOCS = {doc_path.stem: doc_path.read_text() for doc_path in sorted(DOCS_DIR.glob('*.md'))}

print(f'loaded {len(DOCS)} docs:', list(DOCS.keys()))
print(f'total chars: {sum(len(doc_text) for doc_text in DOCS.values())}')
print('prompt files:', sorted(prompt_path.name for prompt_path in PROMPTS_DIR.glob('*.md')))

loaded 11 docs: ['analytics', 'architecture', 'auth', 'availability', 'client_profiles', 'customer_booking_flow', 'onboarding', 'realtime_sync', 'roadmap', 'service_management', 'time_slot_algorithm']
total chars: 12113
prompt files: ['assistant_v1.md', 'judge_v1.md']


### Save model responses so reruns give the same results

Every model response is saved to JSON the first time and reused after that, so reruns make no API calls and scores don't drift. My saved responses are in `week-8/cache/`, so the notebook also runs without an API key.

In [4]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = pathlib.Path('/content/drive/MyDrive/cosc650_week8/cache')
except ImportError:
    CACHE_DIR = pathlib.Path('cache')  # not in Colab: keep it next to the notebook
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# seed the cache with the committed runs from the repo, without overwriting newer ones
REPO_CACHE_DIR = pathlib.Path('_repo/week-8/cache')
if REPO_CACHE_DIR.exists():
    for repo_cache_file in REPO_CACHE_DIR.glob('*.json'):
        local_copy = CACHE_DIR / repo_cache_file.name
        if not local_copy.exists():
            shutil.copy(repo_cache_file, local_copy)


def load_cache(cache_name):
    """Every saved response of one kind ('hyde', 'runs', ...), as a dict keyed by a string describing the call."""
    cache_path = CACHE_DIR / f'{cache_name}.json'
    if not cache_path.exists():
        return {}
    return json.loads(cache_path.read_text())


def save_cache(cache_name, cache):
    """Write one kind of saved response back to disk."""
    cache_path = CACHE_DIR / f'{cache_name}.json'
    cache_path.write_text(json.dumps(cache, indent=2))


print('cache folder:', CACHE_DIR)
print('saved caches:', sorted(cache_file.name for cache_file in CACHE_DIR.glob('*.json')))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cache folder: /content/drive/MyDrive/cosc650_week8/cache
saved caches: ['hyde.json', 'judge.json', 'runs.json']


### Gemini helper for single prompts

Reused from Weeks 5 and 6, for HyDE and the judge. The agent itself uses `google-genai`, the client Week 4's tool loop was built on.

In [5]:
!pip install -q openai

def gemini_chat(messages, model=MODEL, **kw):
    """Gemini via the OpenAI-compatible endpoint. Returns text, or None if no key."""
    key = os.environ.get('GEMINI_API_KEY')
    if not key:
        return None
    from openai import OpenAI
    client = OpenAI(api_key=key, base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

## Retrieval: the `search_docs` tool (Weeks 5 and 6)

### Set up the embedder and cross-encoder

In [6]:
!pip install -q faiss-cpu sentence-transformers

# HF_HOME has to be set before sentence_transformers is imported, or the models download to the default cache instead
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

embedder = SentenceTransformer(EMBED_MODEL_NAME)
reranker = CrossEncoder(RERANK_MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

### Chunk the docs by paragraph and build the index

Week 5's best configuration (recall@3 of 1.00), using Week 5's `chunk_paragraph` and Week 6's `build_chunks` and `build_index`.

In [7]:
def chunk_paragraph(text):
    """Split text into one chunk per paragraph (blank-line-separated)."""
    raw_paragraphs = text.split('\n\n')
    paragraphs = []
    for paragraph in raw_paragraphs:
        cleaned_paragraph = re.sub(r'\s+', ' ', paragraph).strip() # Collapse newlines and repeated spaces into single spaces
        if cleaned_paragraph:  # skip empty paragraphs, e.g. a leading or trailing blank line
            paragraphs.append(cleaned_paragraph)
    return paragraphs


def build_chunks(docs, chunk_fn):
    """
    Chunk each document separately (not one concatenated blob) so a chunk
    never straddles two different docs, and tag each chunk with the doc
    name it came from -- search_docs returns that name so the assistant can
    cite it, and the relevance labeling below needs it too.
    """
    all_chunks = []
    chunk_sources = []
    for doc_name, doc_text in docs.items():
        doc_chunks = chunk_fn(doc_text)
        all_chunks.extend(doc_chunks)
        chunk_sources.extend([doc_name] * len(doc_chunks))
    return all_chunks, chunk_sources


def build_index(chunks):
    """Embed a list of chunks and load them into a FAISS index for similarity search."""
    chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True).astype('float32')
    index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    index.add(chunk_embeddings)
    return index


chunks, chunk_sources = build_chunks(DOCS, chunk_paragraph)
index = build_index(chunks)
print(f"by-paragraph chunking (Week 5's best config) -> {len(chunks)} chunks from {len(DOCS)} docs")

by-paragraph chunking (Week 5's best config) -> 42 chunks from 11 docs


### HyDE, reranking, and the `search_docs` tool

Week 6's HyDE and reranking, each with its own on/off switch so Part 3 can test them separately.

In [8]:
def retrieve_indices(query, k):
    """Return the positions of the top-k chunks most similar to the query."""
    query_embedding = embedder.encode([query], normalize_embeddings=True).astype('float32')
    _, retrieved_positions = index.search(query_embedding, k)
    # FAISS pads with -1 when fewer than k chunks exist in the index -- drop those.
    return [position for position in retrieved_positions[0] if position >= 0]


def hyde_generate(query):
    """
    Ask an LLM to write a short hypothetical passage, in the same
    documentation style as the corpus, that would answer the query -- then
    that passage (not the literal question) is what gets embedded and
    searched. Passages are saved, so the same query always searches the same way.
    """
    hyde_cache = load_cache('hyde')
    cache_key = f'{MODEL}|{query}'
    if cache_key in hyde_cache:
        return hyde_cache[cache_key]

    prompt = (
        'Write a short passage (2-3 sentences) from internal technical '
        'documentation for a nail salon booking app that would answer this '
        'question. Write it as a factual statement, not a question, in a '
        'confident, documentation-like style -- your best guess at the '
        'specific detail, not a hedge.\n\n'
        f'Question: {query}\nPassage:'
    )
    passage = gemini_chat([{'role': 'user', 'content': prompt}], temperature=TEMPERATURE)
    if passage is None:
        return query  # no API key and nothing saved: search with the raw query, as Week 6 did

    hyde_cache[cache_key] = passage
    save_cache('hyde', hyde_cache)
    return passage


def get_score(position_and_score):
    """Sort key for (position, score) pairs."""
    return position_and_score[1]


def rerank(query, candidate_positions, k):
    """
    Score each candidate chunk against the original query with a
    cross-encoder, and return the top-k candidate positions by that score.
    """
    query_chunk_pairs = [(query, chunks[position]) for position in candidate_positions]
    scores = reranker.predict(query_chunk_pairs)
    scored_positions = sorted(zip(candidate_positions, scores), key=get_score, reverse=True)
    best_scored_positions = scored_positions[:k]
    return [position for position, score in best_scored_positions]


def retrieve_positions(query, use_hyde, use_rerank):
    """
    Positions of the TOP_K chunks for this query, using whichever Week 6
    techniques are switched on. With reranking, FAISS pulls a wider pool of
    CANDIDATE_N first, and the cross-encoder scores that pool against the
    ORIGINAL query (not the HyDE passage).
    """
    search_text = query
    if use_hyde:
        search_text = hyde_generate(query)

    if not use_rerank:
        return retrieve_indices(search_text, TOP_K)

    candidate_positions = retrieve_indices(search_text, CANDIDATE_N)
    return rerank(query, candidate_positions, TOP_K)


def search_docs(query):
    """The search_docs tool: the TOP_K passages for the model's query, each tagged with the doc it came from."""
    # USE_HYDE and USE_RERANK are read here, at call time, so flipping them for an ablation takes effect immediately
    positions = retrieve_positions(query, use_hyde=USE_HYDE, use_rerank=USE_RERANK)
    passages = []
    for position in positions:
        passages.append({'doc': chunk_sources[position], 'text': chunks[position]})
    return {'query': query, 'passages': passages}

### Check retrieval on the Week 5 questions

A sanity check on the 11 questions from Weeks 5 and 6, under each retrieval setting. Those questions shaped earlier choices, so Part 2 uses new ones for the real evaluation.

In [9]:
import pandas as pd

def label_relevant(source_doc, fact):
    """
    Find every chunk that counts as ground-truth relevant for one query: it
    has to come from the right document AND actually contain the fact's
    wording.
    """
    fact_lower = fact.lower()
    relevant_positions = []
    for position, (chunk_text, chunk_source) in enumerate(zip(chunks, chunk_sources)):
        if chunk_source == source_doc and fact_lower in chunk_text.lower():
            relevant_positions.append(position)
    return relevant_positions


def precision_recall(retrieved_positions, source_doc, fact):
    """
    Precision is how many of the retrieved chunks were actually relevant;
    recall is how many of the relevant chunks that exist were actually
    retrieved.
    """
    relevant_positions = label_relevant(source_doc, fact)
    hits = [position for position in retrieved_positions if position in relevant_positions]
    precision = len(hits) / len(retrieved_positions) if retrieved_positions else 0.0
    recall = len(hits) / len(relevant_positions) if relevant_positions else 0.0
    return precision, recall


# Query, the fact that answers it, and which doc that fact lives in -- identical to Weeks 5 and 6
WEEK5_QUESTIONS = [
    ('How far in advance must a customer book?', '1 hour', 'time_slot_algorithm'),
    ('What does the owner app use for navigation?', 'Expo Router', 'architecture'),
    ('How do new bookings appear on the dashboard without a refresh?', 'no pull-to-refresh or manual reload needed', 'realtime_sync'),
    ("Where do new owners' service lists start from?", 'default service catalog', 'service_management'),
    ('How many steps are in the customer booking flow?', 'four steps', 'customer_booking_flow'),
    ('What happens to Google sign-in when running in Expo Go?', 'mock version', 'auth'),
    ('What three time ranges can the analytics dashboard filter by?', 'This Week, This Month, and This Year', 'analytics'),
    ('What two fields track onboarding progress?', "dismissed the checklist", 'onboarding'),
    ("How is a client's fill due date determined?", 'computed fill due date', 'client_profiles'),
    ('What limits how many bookings an owner can take in a day?', 'maximum number of bookings allowed per day', 'availability'),
    ('Does the app support SMS booking confirmations today?', 'SMS booking confirmations', 'roadmap'),
]

RETRIEVAL_SETTINGS = {
    'paragraph only': {'use_hyde': False, 'use_rerank': False},
    '+ rerank':       {'use_hyde': False, 'use_rerank': True},
    '+ HyDE + rerank': {'use_hyde': True, 'use_rerank': True},
}


def build_recall_table(questions, retrieval_settings):
    """One row per question, with recall@TOP_K under each retrieval setting side by side."""
    table_rows = []
    for query, fact, source_doc in questions:
        row = {'query': query, 'source doc': source_doc}
        for setting_name, setting in retrieval_settings.items():
            positions = retrieve_positions(query, **setting)
            _, recall = precision_recall(positions, source_doc, fact)
            row[setting_name] = recall
        table_rows.append(row)
    return pd.DataFrame(table_rows)


recall_table = build_recall_table(WEEK5_QUESTIONS, RETRIEVAL_SETTINGS)
print(f'mean recall@{TOP_K}:')
for setting_name in RETRIEVAL_SETTINGS:
    print(f'  {setting_name:16s} {recall_table[setting_name].mean():.2f}')
recall_table

mean recall@3:
  paragraph only   1.00
  + rerank         1.00
  + HyDE + rerank  1.00


,query,source doc,paragraph only,+ rerank,+ HyDE + rerank
0,How far in advance must a customer book?,time_slot_algorithm,1.0,1.0,1.0
1,What does the owner app use for navigation?,architecture,1.0,1.0,1.0
2,How do new bookings appear on the dashboard wi...,realtime_sync,1.0,1.0,1.0
3,Where do new owners' service lists start from?,service_management,1.0,1.0,1.0
4,How many steps are in the customer booking flow?,customer_booking_flow,1.0,1.0,1.0
5,What happens to Google sign-in when running in...,auth,1.0,1.0,1.0
6,What three time ranges can the analytics dashb...,analytics,1.0,1.0,1.0
7,What two fields track onboarding progress?,onboarding,1.0,1.0,1.0
8,How is a client's fill due date determined?,client_profiles,1.0,1.0,1.0
9,What limits how many bookings an owner can tak...,availability,1.0,1.0,1.0


## Tools (Week 4)

### Mock owner data

Stands in for the app's Firestore data, in the same shapes, like Week 4's `FAKE_REPO`. There are two salons so owner scoping can be tested. Service lengths are the app's default catalog from [`services.ts`](https://github.com/BenMillerDev/NailSalonApp/blob/4df756e2504da6ad1782388ca573c330d029238b/src/constants/services.ts).

In [10]:
SERVICE_DURATIONS = {
    'classic-manicure': 45, 'gel-manicure': 60, 'acrylic-full-set': 90,
    'acrylic-fill': 60, 'classic-pedicure': 60, 'gel-pedicure': 75,
    'gel-removal': 30, 'acrylic-removal': 45,
    'lash-classic-full': 120, 'lash-hybrid-full': 135, 'lash-volume-full': 150,
    'lash-classic-fill': 60, 'lash-hybrid-fill': 75, 'lash-volume-fill': 90,
    'lash-removal': 30,
    'brow-wax': 20, 'brow-lamination': 45, 'brow-tint': 20,
}
ADD_ON_DURATIONS = {
    'nail-art': 15, 'ombre': 15, 'chrome-powder': 10, 'paraffin-wax': 15,
    'callus-removal': 15, 'shape-change': 10, 'gel-removal': 30,
    'lash-bath': 10, 'brow-tint': 20, 'brow-wax': 20,
}
NAIL_LENGTH_EXTRA_MINUTES = {'short': 0, 'medium': 0, 'long': 15, 'xl': 20}

# availability/{ownerId}
MOCK_AVAILABILITY_BY_OWNER = {
    'salon-owner-1': {
        'sunday':    {'isOpen': False, 'startTime': '9:00', 'endTime': '18:00'},
        'monday':    {'isOpen': True,  'startTime': '9:00', 'endTime': '18:00'},
        'tuesday':   {'isOpen': True,  'startTime': '9:00', 'endTime': '18:00'},
        'wednesday': {'isOpen': True,  'startTime': '9:00', 'endTime': '18:00'},
        'thursday':  {'isOpen': True,  'startTime': '9:00', 'endTime': '18:00'},
        'friday':    {'isOpen': True,  'startTime': '9:00', 'endTime': '18:00'},
        'saturday':  {'isOpen': True,  'startTime': '9:00', 'endTime': '15:00'},
        'bufferTime': 15,
        'blockedDates': ['2026-10-31'],
    },
    'salon-owner-2': {
        'sunday':    {'isOpen': False, 'startTime': '9:00', 'endTime': '18:00'},
        'monday':    {'isOpen': False, 'startTime': '9:00', 'endTime': '18:00'},
        'tuesday':   {'isOpen': True,  'startTime': '10:00', 'endTime': '19:00'},
        'wednesday': {'isOpen': True,  'startTime': '10:00', 'endTime': '19:00'},
        'thursday':  {'isOpen': True,  'startTime': '10:00', 'endTime': '19:00'},
        'friday':    {'isOpen': True,  'startTime': '10:00', 'endTime': '19:00'},
        'saturday':  {'isOpen': True,  'startTime': '10:00', 'endTime': '16:00'},
        'bufferTime': 30,
        'blockedDates': [],
    },
}

# appointments collection: every document carries its ownerId
MOCK_APPOINTMENTS = [
    {'id': 'a1', 'ownerId': 'salon-owner-1', 'clientName': 'Jordan Lee', 'clientPhone': '555-0101',
     'startTime': '2026-10-17T10:00:00', 'endTime': '2026-10-17T11:00:00', 'status': 'confirmed'},
    {'id': 'a2', 'ownerId': 'salon-owner-1', 'clientName': 'Sam Rivera', 'clientPhone': '555-0102',
     'startTime': '2026-10-17T13:00:00', 'endTime': '2026-10-17T14:30:00', 'status': 'pending'},
    # cancelled, so it must not block any slots
    {'id': 'a3', 'ownerId': 'salon-owner-1', 'clientName': 'Alex Kim', 'clientPhone': '555-0103',
     'startTime': '2026-10-17T12:00:00', 'endTime': '2026-10-17T12:45:00', 'status': 'cancelled'},
    # same day, different salon, so it must never affect salon-owner-1's slots
    {'id': 'a4', 'ownerId': 'salon-owner-2', 'clientName': 'Taylor Brooks', 'clientPhone': '555-0201',
     'startTime': '2026-10-17T11:00:00', 'endTime': '2026-10-17T13:00:00', 'status': 'confirmed'},
]
print('salons:', list(MOCK_AVAILABILITY_BY_OWNER), '| services:', len(SERVICE_DURATIONS),
      '| add-ons:', len(ADD_ON_DURATIONS), '| appointments:', len(MOCK_APPOINTMENTS))

salons: ['salon-owner-1', 'salon-owner-2'] | services: 18 | add-ons: 10 | appointments: 4


### Tool schemas

Week 4's `TOOLS` format, plus two fields only `dispatch()` reads: `access` (every tool is read-only) and `owner_scoped` (`dispatch()` supplies the owner id, never the model). Service ids are listed as the only allowed values, which applies Week 4's path-guessing fix up front.

In [11]:
TOOLS = [
    {
        'name': 'search_docs',
        'description': 'Search the NailSalonApp documentation for how a feature works or why it was built '
                       'that way. Returns the most relevant passages and the doc each one came from.',
        'access': 'read',
        'owner_scoped': False,  # the docs describe the app, not one salon
        'parameters': {
            'type': 'object',
            'properties': {'query': {'type': 'string'}},
            'required': ['query'],
        },
    },
    {
        'name': 'get_salon_settings',
        'description': "Look up this owner's current settings: weekly open hours, buffer time between "
                       'appointments, blocked dates, and the service menu with durations in minutes.',
        'access': 'read',
        'owner_scoped': True,
        # no 'parameters' key: Gemini rejects an object schema with no properties
    },
    {
        'name': 'get_available_slots',
        'description': 'List the open appointment start times on one date for a set of services, '
                       'using the same rules as the booking app. date is YYYY-MM-DD.',
        'access': 'read',
        'owner_scoped': True,
        'parameters': {
            'type': 'object',
            'properties': {
                'date': {'type': 'string'},
                'service_ids': {'type': 'array', 'items': {'type': 'string', 'enum': sorted(SERVICE_DURATIONS)}},
                'add_on_ids': {'type': 'array', 'items': {'type': 'string', 'enum': sorted(ADD_ON_DURATIONS)}},
                'nail_length_id': {'type': 'string', 'enum': list(NAIL_LENGTH_EXTRA_MINUTES)},
            },
            'required': ['date', 'service_ids'],
        },
    },
    {
        'name': 'run_python',
        'description': 'Run a single allowlisted arithmetic expression, e.g. to add up durations '
                       'or count hours (guarded code-runner).',
        'access': 'read',
        'owner_scoped': False,
        'parameters': {
            'type': 'object',
            'properties': {'expression': {'type': 'string'}},
            'required': ['expression'],
        },
    },
]
print('tools:', [tool['name'] for tool in TOOLS])

tools: ['search_docs', 'get_salon_settings', 'get_available_slots', 'run_python']


### Time slot tool, ported from NailSalonApp's `timeSlots.ts`

A Python port of [`timeSlots.ts`](https://github.com/BenMillerDev/NailSalonApp/blob/4df756e2504da6ad1782388ca573c330d029238b/src/utils/timeSlots.ts) and the booking site's [`getBookingsForDate`](https://github.com/BenMillerDev/NailSalonApp/blob/4df756e2504da6ad1782388ca573c330d029238b/booking-web/src/services/bookingService.ts), so the assistant gives the same times the app would. Differences: Firestore reads become lookups into the mock data, "now" is fixed for reproducible results, it takes ids instead of `Service` objects, and there's no daily booking cap.

In [12]:
def get_availability(owner_id):
    """One salon's availability document (availability/{ownerId} in Firestore)."""
    if owner_id not in MOCK_AVAILABILITY_BY_OWNER:
        raise ValueError('no availability settings found for this salon')
    return MOCK_AVAILABILITY_BY_OWNER[owner_id]


def get_bookings_for_date(owner_id, day):
    """
    Port of getBookingsForDate: this salon's appointments on this day that still
    hold their time. Like the app's query, it returns only id and times, so client
    names and phone numbers never reach the slot math or the model.
    """
    bookings = []
    for appointment in MOCK_APPOINTMENTS:
        is_this_salon = appointment['ownerId'] == owner_id
        is_this_day = datetime.fromisoformat(appointment['startTime']).date() == day.date()
        holds_its_time = appointment['status'] in BOOKED_STATUSES
        if is_this_salon and is_this_day and holds_its_time:
            bookings.append({'id': appointment['id'],
                             'startTime': appointment['startTime'],
                             'endTime': appointment['endTime']})
    return bookings


def get_day_name(day):
    """Return the lowercase weekday name ('monday', ...) used as the availability key."""
    return day.strftime('%A').lower()


def parse_time_string(day, time_string):
    """Combine a date with an 'H:MM' string from the owner's hours into a datetime."""
    hours, minutes = time_string.split(':')
    return datetime(day.year, day.month, day.day, int(hours), int(minutes))


def get_blocked_ranges(day, bookings, buffer_minutes):
    """
    Turn this day's existing bookings into blocked (start, end) ranges, padded
    with the buffer on BOTH sides -- so a new appointment can't start right
    after, or end right before, an existing one.
    """
    buffer = timedelta(minutes=buffer_minutes)
    blocked_ranges = []
    for booking in bookings:
        booking_start = datetime.fromisoformat(booking['startTime'])
        booking_end = datetime.fromisoformat(booking['endTime'])
        if booking_start.date() == day.date():
            blocked_ranges.append((booking_start - buffer, booking_end + buffer))
    return blocked_ranges


def overlaps_with_blocked(slot_start, slot_end, blocked_ranges):
    """
    True if the slot overlaps any blocked range. Comparisons are strict, so a
    slot that ends exactly when a blocked range begins is still allowed.
    """
    for range_start, range_end in blocked_ranges:
        if slot_start < range_end and slot_end > range_start:
            return True
    return False


def get_available_slots(day, total_duration, availability, existing_bookings, now=NOW):
    """
    Every start time, in 15-minute steps, where the full appointment fits inside
    the owner's hours without overlapping a buffered existing booking.
    """
    # Step 1 -- the owner isn't working this day of the week
    day_availability = availability.get(get_day_name(day))
    if day_availability is None or not day_availability['isOpen']:
        return []

    # Step 2 -- the owner manually blocked this specific date
    # (timeSlots.ts checks this after building blocked ranges; same result)
    if day.strftime('%Y-%m-%d') in availability.get('blockedDates', []):
        return []

    # Step 3 -- existing bookings, padded with the owner's buffer
    buffer_minutes = availability.get('bufferTime', DEFAULT_BUFFER_MINUTES)
    blocked_ranges = get_blocked_ranges(day, existing_bookings, buffer_minutes)

    # Step 4 -- walk every candidate start time across the work day
    work_start = parse_time_string(day, day_availability['startTime'])
    work_end = parse_time_string(day, day_availability['endTime'])
    appointment_length = timedelta(minutes=total_duration)
    step = timedelta(minutes=SLOT_INCREMENT_MINUTES)

    slots = []
    current = work_start
    while current < work_end:
        slot_end = current + appointment_length
        fits_before_close = slot_end <= work_end
        if fits_before_close and not overlaps_with_blocked(current, slot_end, blocked_ranges):
            slots.append(current)
        current = current + step

    # Step 5 -- same-day bookings must start more than 1 hour from now
    if day.date() == now.date():
        minimum_start = now + timedelta(minutes=MINIMUM_ADVANCE_MINUTES)
        slots = [slot for slot in slots if slot > minimum_start]

    return slots


def calculate_total_duration(service_ids, add_on_ids=None, nail_length_id=None):
    """Total appointment length in minutes: services + add-ons + any extra time for longer nails."""
    add_on_ids = add_on_ids or []  # add-ons are optional, so the model may leave them out

    services_minutes = 0
    for service_id in service_ids:
        services_minutes += SERVICE_DURATIONS[service_id]

    add_ons_minutes = 0
    for add_on_id in add_on_ids:
        add_ons_minutes += ADD_ON_DURATIONS[add_on_id]

    length_minutes = NAIL_LENGTH_EXTRA_MINUTES.get(nail_length_id, 0)  # no length chosen = no extra time
    return services_minutes + add_ons_minutes + length_minutes


def format_time_slot(slot):
    """datetime -> '2:30 PM', the same format the app shows customers."""
    return slot.strftime('%I:%M %p').lstrip('0')


def get_available_slots_tool(owner_id, date, service_ids, add_on_ids=None, nail_length_id=None):
    """
    What dispatch() calls for get_available_slots. owner_id comes from dispatch(),
    not the model; everything else is the model's plain JSON arguments. Returns
    something JSON-serializable. A malformed date raises ValueError, which
    dispatch() reports back to the model as an execution_error.
    """
    day = datetime.strptime(date, '%Y-%m-%d')
    total_duration = calculate_total_duration(service_ids, add_on_ids, nail_length_id)
    availability = get_availability(owner_id)
    existing_bookings = get_bookings_for_date(owner_id, day)
    slots = get_available_slots(day, total_duration, availability, existing_bookings)
    return {
        'date': date,
        'weekday': get_day_name(day),
        'total_duration_minutes': total_duration,
        'slot_count': len(slots),
        'slots': [format_time_slot(slot) for slot in slots],
    }

### Salon settings and guarded calculator tools

`run_python` is Week 4's guarded calculator. `get_salon_settings` is a plain lookup, like Week 4's `get_file_stats`, for one salon only.

In [13]:
ALLOWED_OPERATORS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg,
}


def get_operator_function(operator_node):
    """The Python function for an allowlisted operator, or a clear error for any other."""
    operator_type = type(operator_node)
    if operator_type not in ALLOWED_OPERATORS:
        raise ValueError(f'operator not allowed: {operator_type.__name__}')
    return ALLOWED_OPERATORS[operator_type]


def evaluate_arithmetic(node):
    """
    Compute a parsed expression one piece at a time, allowing only numbers and
    the operators above. Anything else -- names, function calls, imports,
    attribute access -- raises, which is what makes this code-runner safe.
    """
    is_number = isinstance(node, ast.Constant) and type(node.value) in (int, float)
    if is_number:
        return node.value
    if isinstance(node, ast.BinOp):  # e.g. 60 + 15
        apply_operator = get_operator_function(node.op)
        return apply_operator(evaluate_arithmetic(node.left), evaluate_arithmetic(node.right))
    if isinstance(node, ast.UnaryOp):  # e.g. -15
        apply_operator = get_operator_function(node.op)
        return apply_operator(evaluate_arithmetic(node.operand))
    raise ValueError('only arithmetic is allowed')


class ExpressionTimeout(Exception): pass


def raise_timeout(signal_number, stack_frame):
    """Called by the alarm signal when an expression runs too long (e.g. a huge power like 9**9**9)."""
    raise ExpressionTimeout('expression took too long to evaluate')


def run_python(expression, timeout_seconds=2):
    """The run_python tool: evaluate one arithmetic expression, giving up after timeout_seconds."""
    signal.signal(signal.SIGALRM, raise_timeout)
    signal.alarm(timeout_seconds)
    try:
        parsed_expression = ast.parse(expression, mode='eval')
        return evaluate_arithmetic(parsed_expression.body)
    finally:
        signal.alarm(0)  # cancel so a stale alarm can't fire on some later, unrelated call


def get_salon_settings(owner_id):
    """
    One salon's settings as the model needs them: hours, buffer, blocked dates,
    and service lengths. owner_id comes from dispatch(), not the model.
    """
    availability = get_availability(owner_id)
    weekdays = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']
    weekly_hours = {}
    for day_name in weekdays:
        weekly_hours[day_name] = availability[day_name]
    return {
        'weekly_hours': weekly_hours,
        'buffer_minutes': availability['bufferTime'],
        'blocked_dates': availability['blockedDates'],
        'service_minutes': SERVICE_DURATIONS,
        'add_on_minutes': ADD_ON_DURATIONS,
        'nail_length_extra_minutes': NAIL_LENGTH_EXTRA_MINUTES,
    }

# tool name -> the Python function dispatch() calls for it
IMPL = {
    'search_docs': search_docs,
    'get_salon_settings': get_salon_settings,
    'get_available_slots': get_available_slots_tool,
    'run_python': run_python,
}

### Argument validation

Week 4's `validate()`, split into one small check per argument type. New this week: `check_list_of_ids`, for the list arguments.

In [14]:
class ToolArgError(Exception): pass       # the model sent bad arguments; it can fix them and retry
class ToolNotPermitted(Exception): pass   # the tool isn't allowed to run at all


def find_tool(tool_name):
    """The TOOLS entry for this name. An unknown name is the model's mistake, so it's a ToolArgError."""
    for tool in TOOLS:
        if tool['name'] == tool_name:
            return tool
    raise ToolArgError(f'unknown tool: {tool_name}')


def check_string(field_name, value):
    """Raise unless the value is a string."""
    if not isinstance(value, str):
        raise ToolArgError(f'{field_name} must be a string')


def check_number(field_name, value):
    """Raise unless the value is a real, finite number."""
    # type() rather than isinstance(), because Python counts True and False as ints
    if type(value) not in (int, float):
        raise ToolArgError(f'{field_name} must be a number (not a boolean)')
    if isinstance(value, float) and not math.isfinite(value):
        raise ToolArgError(f'{field_name} must be finite')


def check_list_of_ids(field_name, value, item_spec):
    """
    New in Week 8: raise unless the value is a list of string ids from the
    schema's allowed list. Lists of ids are the only arrays TOOLS uses.
    """
    if item_spec['type'] != 'string':
        raise ValueError(f'extend validate() to support arrays of: {item_spec["type"]}')
    if not isinstance(value, list):
        raise ToolArgError(f'{field_name} must be a list')

    allowed_ids = item_spec.get('enum')
    for item in value:
        if not isinstance(item, str):
            raise ToolArgError(f'every item in {field_name} must be a string')
        if allowed_ids is not None and item not in allowed_ids:
            raise ToolArgError(f'{field_name} item {item!r} not in {allowed_ids}')


def check_field(field_name, value, field_spec):
    """Check one argument against its schema: the type first, then the allowed values if the schema lists them."""
    field_type = field_spec['type']
    if field_type == 'string':
        check_string(field_name, value)
    elif field_type == 'number':
        check_number(field_name, value)
    elif field_type == 'array':
        check_list_of_ids(field_name, value, field_spec['items'])
    else:
        raise ValueError(f'extend validate() to support schema type: {field_type}')

    allowed_values = field_spec.get('enum')
    if allowed_values is not None and value not in allowed_values:
        raise ToolArgError(f'{field_name}={value!r} not in {allowed_values}')


def validate(tool, arguments):
    """Raise ToolArgError if the model's arguments don't match this tool's schema."""
    no_arguments_schema = {'type': 'object', 'properties': {}}
    schema = tool.get('parameters', no_arguments_schema)  # get_salon_settings has no parameters
    if not isinstance(arguments, dict):
        raise ToolArgError('arguments must be a JSON object')

    for required_field in schema.get('required', []):
        if required_field not in arguments:
            raise ToolArgError(f'missing required field: {required_field}')

    for field_name, value in arguments.items():
        field_spec = schema['properties'].get(field_name)
        if field_spec is None:
            # this is also what stops a model-supplied owner_id: no schema declares one
            raise ToolArgError(f'unexpected field: {field_name}')
        check_field(field_name, value, field_spec)

### Dispatch with the safety rules

Week 4's `dispatch()`, plus two rules: it only runs tools marked read-only, and it adds the signed-in owner's id itself, so the model can't pick a salon.

In [15]:
def check_permitted(tool):
    """Read-only for now: raise unless the tool is explicitly marked 'access': 'read'."""
    is_read_only = tool.get('access') == 'read'
    if not is_read_only and not ALLOW_WRITE_TOOLS:
        raise ToolNotPermitted(f"{tool['name']} can change data, and this assistant is read-only")


def run_tool(tool, arguments, owner_id):
    """Call the tool's function, adding owner_id for tools that read one salon's data."""
    tool_function = IMPL[tool['name']]
    if tool['owner_scoped']:
        return tool_function(owner_id=owner_id, **arguments)
    return tool_function(**arguments)


def dispatch(tool_name, arguments, owner_id):
    """
    Run one tool call from the model and return a result it can read, success or
    failure. owner_id has no default on purpose: every call has to say which
    signed-in owner it's acting for.
    """
    try:
        tool = find_tool(tool_name)
        check_permitted(tool)
        validate(tool, arguments)
        output = run_tool(tool, arguments, owner_id)
        json.dumps(output, allow_nan=False)  # reject results the model cannot receive as JSON
        return {'ok': True, 'tool': tool_name, 'output': output}
    except ToolArgError as error:
        return {'ok': False, 'tool': tool_name, 'error_type': 'invalid_arguments', 'message': str(error)}
    except ToolNotPermitted as error:
        return {'ok': False, 'tool': tool_name, 'error_type': 'not_permitted', 'message': str(error)}
    except Exception as error:
        return {'ok': False, 'tool': tool_name, 'error_type': 'execution_error', 'message': str(error)}

### Sanity-check the tools without the model

Scripted calls, like Week 4's, check every tool and error path before the model is involved. The expected results are hand-calculated and noted beside each call. The last three check the safety rules: the second salon gets its own answer, and the model can't name a salon or reach one that doesn't exist.

In [16]:
scripted = [
  ('search_docs',         {'query':'How far in advance must a customer book?'},                 OWNER_ID),  # expect time_slot_algorithm
  ('get_available_slots', {'date':'2026-10-17','service_ids':['gel-manicure']},                 OWNER_ID),  # expect 11:15, 11:30, 11:45 AM
  ('get_available_slots', {'date':'2026-10-17','service_ids':['gel-manicure'],
                           'add_on_ids':['nail-art'],'nail_length_id':'long'},                  OWNER_ID),  # 90 min fills 11:15-12:45 exactly: expect 11:15 AM only
  ('get_available_slots', {'date':'2026-10-18','service_ids':['gel-manicure']},                 OWNER_ID),  # Sunday, closed
  ('get_available_slots', {'date':'2026-10-31','service_ids':['gel-manicure']},                 OWNER_ID),  # blocked date
  ('get_available_slots', {'date':'2026-10-12','service_ids':['classic-manicure']},             OWNER_ID),  # today: nothing before 11:15
  ('get_available_slots', {'date':'2026-10-17','service_ids':['gel-mani']},                     OWNER_ID),  # scripted invalid id
  ('get_available_slots', {'date':'2026-10-17','service_ids':'gel-manicure'},                   OWNER_ID),  # scripted string, not a list
  ('get_available_slots', {'date':'10/17/2026','service_ids':['gel-manicure']},                 OWNER_ID),  # scripted wrong date format
  ('get_salon_settings',  {},                                                                    OWNER_ID),
  ('run_python',          {'expression':'60+15+15'},                                             OWNER_ID),
  ('run_python',          {'expression':'__import__("os").system("echo hi")'},                  OWNER_ID),  # guarded
  # safety rules
  ('get_available_slots', {'date':'2026-10-17','service_ids':['gel-manicure']},                 'salon-owner-2'),  # expect 1:30 PM to 3:00 PM
  ('get_salon_settings',  {'owner_id':'salon-owner-2'},                                          OWNER_ID),         # model tries to pick the salon
  ('get_salon_settings',  {},                                                                    'salon-owner-9'),  # no such salon
]
for tool_name, arguments, owner_id in scripted:
    result = dispatch(tool_name, arguments, owner_id=owner_id)
    status = 'OK ' if result['ok'] else 'ERR'
    print(f'[{status}] as {owner_id}: {tool_name}({arguments}) -> {json.dumps(result, allow_nan=False)}')

[OK ] search_docs({'query': 'How far in advance must a customer book?'}) -> {"ok": true, "tool": "search_docs", "output": {"query": "How far in advance must a customer book?", "passages": [{"doc": "time_slot_algorithm", "text": "On top of all that, there's a fixed rule that applies no matter what the owner's settings say: a customer must book at least 1 hour in advance. Any slot that falls inside that 1-hour window from the current time is excluded from the results, even if it would otherwise be open, to avoid a booking landing on the owner's calendar with almost no notice."}, {"doc": "time_slot_algorithm", "text": "The logic that decides which appointment times are actually bookable lives in a single shared utility module used by both the owner app and the customer booking web app, so the two surfaces never disagree about what's available. Given a day, it produces the list of open slots a customer is allowed to pick from."}, {"doc": "customer_booking_flow", "text": "## Customer Bookin

## Agent: tools + system prompt (Weeks 3 and 4)

### Load the versioned system prompt

Week 3's pattern: a versioned prompt file with few-shot examples, asking for a JSON answer with its sources and rationale. The examples' questions are kept out of the Part 2 evaluation.

In [18]:
def load_prompt(prompt_name, version):
    # prompt files are the actual versioned artifact -- read from disk the
    # same way any other file under version control would be
    prompt_path = PROMPTS_DIR / f'{prompt_name}_{version}.md'
    return prompt_path.read_text()


def build_system_prompt(prompt_file_text, now):
    """
    Keep everything from the '## System' heading down (the notes above it are
    for people reading the repo, not the model), and fill in today's date so
    the model can turn 'this Saturday' into a real date.
    """
    _, system_prompt = prompt_file_text.split('## System', 1)
    today_text = now.strftime('%A, %B %d, %Y, %I:%M %p')  # e.g. Monday, October 12, 2026, 10:00 AM
    return system_prompt.strip().replace('{today}', today_text)


SYSTEM_PROMPT = build_system_prompt(load_prompt('assistant', PROMPT_VERSION), NOW)
print(f'{PROMPT_VERSION} system prompt: {len(SYSTEM_PROMPT)} chars')
print()
print('--- first 400 chars ---')
print(SYSTEM_PROMPT[:400])

v1 system prompt: 3298 chars

--- first 400 chars ---
You are a support assistant for a nail salon owner who uses NailSalonApp.
Today is Monday, October 12, 2026, 10:00 AM.

Use the tools to answer. Each kind of question has its own tool:

- How the app works, or why it was built a certain way: `search_docs`.
- This owner's own settings (open hours, buffer time, blocked dates, service
  lengths): `get_salon_settings`.
- Open appointment times on a da


### Connect the tools to Gemini

Reused from Week 4. Gemini sees only each tool's name, description and parameters.

In [19]:
!pip install -q google-genai
from google import genai
from google.genai import types

gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY']) if HAS_API_KEY else None


def build_function_declaration(tool):
    """One TOOLS entry as Gemini sees it."""
    return types.FunctionDeclaration(
        name=tool['name'],
        description=tool['description'],
        parameters=tool.get('parameters'),  # None for get_salon_settings, which takes no arguments
    )


function_declarations = [build_function_declaration(tool) for tool in TOOLS]
gemini_tool = types.Tool(function_declarations=function_declarations)
print('gemini tool built from', len(gemini_tool.function_declarations), 'schemas; client ready:', gemini_client is not None)

gemini tool built from 4 schemas; client ready: True


### Agent loop

Week 4's `run_agent`, now sending the system prompt and temperature, and passing the owner id to `dispatch()`.

In [20]:
def _ask_model(contents):
    # Send the conversation so far to Gemini and return its response for
    # this turn. Tool results already appended to `contents` are just more
    # turns in the same history -- Gemini doesn't need them called out
    # specially.
    return gemini_client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=TEMPERATURE,
            tools=[gemini_tool],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        ),
    )


def _requested_calls(candidate):
    # The tool calls Gemini proposed this turn, if any -- empty once it's
    # ready to give a final answer instead.
    if candidate.content is None or not candidate.content.parts:
        return []
    return [part.function_call for part in candidate.content.parts if part.function_call]


def _run_requested_call(call, turn, owner_id, log, verbose):
    # Execute one proposed call through dispatch(), record it in `log`,
    # and return it as a FunctionResponse Part carrying the same call id Gemini
    # sent -- that id is what lets Gemini match this result back to its call.
    args = call.args or {}
    result = dispatch(call.name, args, owner_id)
    log.append({'turn': turn, 'call_id': call.id, 'name': call.name, 'args': args, 'result': result})
    if verbose:
        status = 'OK ' if result['ok'] else 'ERR'
        print(f'[turn {turn}] [{status}] {call.name}({args}) -> {result}')
    return types.Part(function_response=types.FunctionResponse(id=call.id, name=call.name, response=result))


def run_agent(query, owner_id, max_turns=MAX_TURNS, verbose=True):
    # Send `query` to Gemini with TOOLS attached, intercept every tool
    # call it makes, run each one through dispatch() as `owner_id`, and feed
    # the results back -- tagged with the model's own call ids -- until it
    # gives a final text answer (or max_turns is hit).
    contents = [types.Content(role='user', parts=[types.Part(text=query)])]
    log = []

    for turn in range(max_turns):
        response = _ask_model(contents)
        candidate = response.candidates[0]
        contents.append(candidate.content)  # the model's own turn, calls included

        calls = _requested_calls(candidate)
        if not calls:
            final_text = response.text or ''
            log.append({'turn': turn, 'final_text': final_text})
            if verbose: print(f'[turn {turn}] final answer: {final_text}')
            return final_text, log

        tool_response_parts = [_run_requested_call(call, turn, owner_id, log, verbose) for call in calls]
        contents.append(types.Content(role='user', parts=tool_response_parts))

    final_text = f'[gave up: no final answer after {max_turns} turns]'
    log.append({'turn': max_turns, 'gave_up': True, 'final_text': final_text})
    if verbose: print(f'[gave up] no final answer after {max_turns} turns')
    return final_text, log

### Parse the JSON answer

Week 3's `extract_json_object`. A reply that isn't valid JSON is flagged instead of crashing the run.

In [21]:
def extract_json_object(text):
    # models sometimes wrap the JSON in a ```json fence, or add prose before or
    # after it, even when told not to -- pull out just the {...} object so
    # json.loads can parse it regardless of what's around it
    fenced_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if fenced_match:
        return fenced_match.group(1)
    braces_match = re.search(r'\{.*\}', text, re.DOTALL)
    if braces_match:
        return braces_match.group(0)
    return text


def parse_answer(final_text):
    """The model's final reply as answer, sources and rationale, plus a parse_error flag."""
    try:
        parsed_reply = json.loads(extract_json_object(final_text))
    except json.JSONDecodeError:
        parsed_reply = None

    if not isinstance(parsed_reply, dict):
        return {'answer': final_text, 'sources': [], 'rationale': '', 'parse_error': True}

    return {
        'answer': parsed_reply.get('answer', ''),
        'sources': parsed_reply.get('sources', []),
        'rationale': parsed_reply.get('rationale', ''),
        'parse_error': False,
    }

### Ask the assistant, reusing saved runs

`ask_assistant` runs a question, or loads its saved run if one exists for the current settings. `show_run` prints the tool calls and the answer.

In [22]:
def run_cache_key(question, owner_id):
    """Every setting that can change a run's result, joined into one string."""
    settings = [MODEL, PROMPT_VERSION, f'hyde={USE_HYDE}', f'rerank={USE_RERANK}', owner_id, question]
    # Chunking became a setting in Part 3. Every run saved before that used
    # 'paragraph', so it's only added to the key when it's something else --
    # that keeps the Part 1 and Part 2 runs valid instead of rerunning them.
    if CHUNKING != 'paragraph':
        settings.insert(4, f'chunking={CHUNKING}')
    return '|'.join(settings)


def ask_assistant(question, owner_id):
    """
    One full assistant run: the saved one if it exists, otherwise a live run that
    gets saved. Returns None when there's no API key and nothing saved.
    """
    run_cache = load_cache('runs')
    cache_key = run_cache_key(question, owner_id)
    if cache_key in run_cache:
        return run_cache[cache_key]
    if not HAS_API_KEY:
        return None

    final_text, call_log = run_agent(question, owner_id, verbose=False)
    parsed_answer = parse_answer(final_text)
    run = {
        'question': question,
        'owner_id': owner_id,
        'answer': parsed_answer['answer'],
        'sources': parsed_answer['sources'],
        'rationale': parsed_answer['rationale'],
        'parse_error': parsed_answer['parse_error'],
        'final_text': final_text,
        'call_log': call_log,
    }
    # round-trip through JSON (default=str turns any unusual SDK type into text) so a
    # live run looks exactly like one loaded from the cache later
    run = json.loads(json.dumps(run, default=str))
    run_cache[cache_key] = run
    save_cache('runs', run_cache)
    return run


def summarize_tool_result(result):
    """A one-line view of a tool result, short enough to scan in a log."""
    if not result['ok']:
        return f"{result['error_type']}: {result['message']}"
    output = result['output']
    if result['tool'] == 'search_docs':
        retrieved_docs = [passage['doc'] for passage in output['passages']]
        return f'passages from {retrieved_docs}'
    if result['tool'] == 'get_available_slots':
        return f"{output['slot_count']} slots on {output['date']} ({output['total_duration_minutes']} min): {output['slots']}"
    if result['tool'] == 'get_salon_settings':
        return 'hours, buffer, blocked dates, and service lengths'
    return str(output)


def show_run(run):
    """Print one run: the question, every tool call in order, then the answer, sources and rationale."""
    print('QUESTION:', run['question'])
    for log_entry in run['call_log']:
        if 'name' not in log_entry:
            continue  # the final-answer entry; the parsed answer is printed below
        status = 'OK ' if log_entry['result']['ok'] else 'ERR'
        print(f"  [turn {log_entry['turn']}] [{status}] {log_entry['name']}({log_entry['args']})")
        print(f"      -> {summarize_tool_result(log_entry['result'])}")
    print('ANSWER:', textwrap.fill(run['answer'], width=100, subsequent_indent='        '))
    print('SOURCES:', run['sources'])
    print('RATIONALE:', run['rationale'])
    if run['parse_error']:
        print('(reply was not valid JSON; the raw text is shown as the answer)')

### Example questions

Four demonstrations, each using a different part of the system. Part 2's evaluation questions are separate.

| Question | Tools | Expected |
|---|---|---|
| Saturday gel manicure with nail art; why not right after 10:00? | slots, plus settings or docs | 11:15 and 11:30 AM; 15-minute buffer |
| Saturday hours and blocked dates | settings | 9:00–3:00; October 31 blocked |
| Text message confirmations | docs | Planned, not built yet |
| Block off November 2 | none | Can't make changes |

In [23]:
EXAMPLE_QUESTIONS = [
    "Do I have any openings this Saturday for a gel manicure with nail art? And why can't anyone book right after my 10 o'clock?",
    "What are my hours on Saturdays, and do I have any days blocked off this month?",
    "Can my clients get a text message confirmation when they book?",
    "Please block off November 2nd for me, I'll be out of town.",
]

example_runs = []
for question in EXAMPLE_QUESTIONS:
    run = ask_assistant(question, owner_id=OWNER_ID)
    if run is None:
        print('[API-BLOCKED] no saved run for:', question)
        continue
    example_runs.append(run)
    show_run(run)
    print('-' * 100)

QUESTION: Do I have any openings this Saturday for a gel manicure with nail art? And why can't anyone book right after my 10 o'clock?
  [turn 0] [OK ] get_available_slots({'service_ids': ['gel-manicure'], 'date': '2026-10-17', 'add_on_ids': ['nail-art']})
      -> 2 slots on 2026-10-17 (75 min): ['11:15 AM', '11:30 AM']
  [turn 0] [OK ] get_salon_settings({})
      -> hours, buffer, blocked dates, and service lengths
ANSWER: Yes, this Saturday, October 17, you have two openings for a gel manicure with nail art at 11:15 AM
        and 11:30 AM. No one can book right after your 10 o'clock appointment because you have a
        15-minute buffer time setting between appointments.
SOURCES: ['get_available_slots', 'get_salon_settings']
RATIONALE: get_available_slots returned 11:15 AM and 11:30 AM as the available slots for Saturday, and get_salon_settings shows a buffer_minutes setting of 15.
----------------------------------------------------------------------------------------------------

# Part 2: Evaluate
Include an evaluation section with concrete metrics. Add an LLM-as-judge pass that scores the system's output on a small held-out set of 10 to 20 cases against two or three named dimensions, for example correctness and grounding, and report the results.

### Held-out evaluation cases

18 new questions, kept separate from everything used to build the system. Five should be declined.

| Category | Cases | A good answer |
|---|---|---|
| `docs` | 6 | answers from the docs |
| `slots` | 5 | looks up the right date and services |
| `settings` | 2 | answers from the owner's settings |
| `roadmap` | 2 | says it's planned, not available |
| `not_covered` | 2 | says the docs don't cover it |
| `refuse` | 1 | says it can't make changes |

Reference answers were worked out by hand, not taken from the tool. The slot cases:

| Case | Day, length | Expected |
|---|---|---|
| 7 | Tuesday, 90 min | 9:00 AM is open (no bookings) |
| 8 | Saturday, 150 min | Nothing fits (longest gap is 90 min) |
| 9 | Today, 75 min | 11:15 AM–4:45 PM (1-hour notice) |
| 10 | Friday, 65 min | 9:00 AM–4:45 PM |
| 11 | Saturday, salon 2, 60 min | 1:30–3:00 PM (30-minute buffer) |

In [24]:
EVAL_CASES = [
    # --- docs: answered from the documentation ---
    {'id': 1, 'category': 'docs', 'owner_id': OWNER_ID,
     'question': "If I delete all of a client's appointments, do I lose the notes I wrote about her?",
     'reference_answer': "No. Client notes are stored separately from appointments, in their own clientNotes collection keyed by your owner id and the client's email, so they stay even if every appointment for that client is deleted.",
     'should_decline': False, 'required_tools': [['search_docs']], 'expected_slot_call': None,
     'source_doc': 'client_profiles', 'fact': 'clientNotes collection',
     'key_facts': [['separate', 'clientnotes', 'own collection']]},
    {'id': 2, 'category': 'docs', 'owner_id': OWNER_ID,
     'question': 'What does a client see right after they submit a booking?',
     'reference_answer': 'A confirmation screen that shows a full summary of what they just booked.',
     'should_decline': False, 'required_tools': [['search_docs']], 'expected_slot_call': None,
     'source_doc': 'customer_booking_flow', 'fact': 'confirmation screen',
     'key_facts': [['confirmation'], ['summary']]},
    {'id': 3, 'category': 'docs', 'owner_id': OWNER_ID,
     'question': 'If I stop offering a service and hide it, does its booking history get deleted?',
     'reference_answer': 'No. Each service has an isActive flag, so you can hide a service without deleting its history.',
     'should_decline': False, 'required_tools': [['search_docs']], 'expected_slot_call': None,
     'source_doc': 'service_management', 'fact': 'isActive flag',
     'key_facts': [['isactive', 'without deleting']]},
    {'id': 4, 'category': 'docs', 'owner_id': OWNER_ID,
     'question': 'Do my analytics numbers come from a separate reporting service?',
     'reference_answer': "No. The analytics figures are computed client-side, directly from your appointment data in Firestore. There's no separate analytics backend, no overnight job, and no third-party analytics service.",
     'should_decline': False, 'required_tools': [['search_docs']], 'expected_slot_call': None,
     'source_doc': 'analytics', 'fact': 'computed client-side',
     'key_facts': [['client-side', 'client side', 'directly from']]},
    {'id': 5, 'category': 'docs', 'owner_id': OWNER_ID,
     'question': 'Can I share my booking link again after onboarding, and where would I do it?',
     'reference_answer': "Yes. Sharing your booking link is a one-tap action on the settings screen, and it's available any time, not just during onboarding.",
     'should_decline': False, 'required_tools': [['search_docs']], 'expected_slot_call': None,
     'source_doc': 'onboarding', 'fact': 'settings screen',
     'key_facts': [['settings']]},
    {'id': 6, 'category': 'docs', 'owner_id': OWNER_ID,
     'question': 'What details does the app keep on my owner account?',
     'reference_answer': "Your email, your name, your salon's name, your phone number, a role field, and when the account was created. It's stored in Firestore under the users collection, keyed by your Firebase uid.",
     'should_decline': False, 'required_tools': [['search_docs']], 'expected_slot_call': None,
     'source_doc': 'auth', 'fact': "salon's name",
     'key_facts': [['email'], ['salon'], ['phone']]},

    # --- slots: need get_available_slots with the right date and services ---
    {'id': 7, 'category': 'slots', 'owner_id': OWNER_ID,
     'question': 'Can a client come in tomorrow at 9 AM for an acrylic full set?',
     'reference_answer': 'Yes. Tomorrow, Tuesday, October 13, 9:00 AM is open for an acrylic full set, which takes 90 minutes.',
     'should_decline': False, 'required_tools': [['get_available_slots']],
     'expected_slot_call': {'date': '2026-10-13', 'ids': ['acrylic-full-set']},
     'source_doc': None, 'fact': None,
     'key_facts': [['yes'], ['9:00', '9 am']]},
    {'id': 8, 'category': 'slots', 'owner_id': OWNER_ID,
     'question': "What's the latest I could start a volume lash full set this Saturday?",
     'reference_answer': "There's no time that works this Saturday. A volume full set takes 150 minutes, and the longest open stretch that day is 90 minutes (11:15 AM to 12:45 PM), so no start time is available.",
     'should_decline': False, 'required_tools': [['get_available_slots']],
     'expected_slot_call': {'date': '2026-10-17', 'ids': ['lash-volume-full']},
     'source_doc': None, 'fact': None,
     'key_facts': []},
    {'id': 9, 'category': 'slots', 'owner_id': OWNER_ID,
     'question': 'Do I have any openings today for a gel pedicure?',
     'reference_answer': "Yes. A gel pedicure (75 minutes) can start any time from 11:15 AM to 4:45 PM today, in 15-minute steps. Nothing earlier works because same-day bookings need at least an hour's notice.",
     'should_decline': False, 'required_tools': [['get_available_slots']],
     'expected_slot_call': {'date': '2026-10-12', 'ids': ['gel-pedicure']},
     'source_doc': None, 'fact': None,
     'key_facts': [['11:15'], ['4:45']]},
    {'id': 10, 'category': 'slots', 'owner_id': OWNER_ID,
     'question': 'A client wants a classic manicure and a brow wax back to back on Friday the 16th. What times work?',
     'reference_answer': 'Together they take 65 minutes, so any start time from 9:00 AM to 4:45 PM on Friday, October 16 works, in 15-minute steps.',
     'should_decline': False, 'required_tools': [['get_available_slots']],
     # brow-wax is both a service and an add-on id, so either split counts as the same lookup
     'expected_slot_call': {'date': '2026-10-16', 'ids': ['brow-wax', 'classic-manicure']},
     'source_doc': None, 'fact': None,
     'key_facts': [['9:00', '9 am'], ['4:45']]},
    {'id': 11, 'category': 'slots', 'owner_id': 'salon-owner-2',
     'question': 'Any gel manicure openings this Saturday?',
     'reference_answer': 'Yes. This Saturday, October 17, a gel manicure can start any time from 1:30 PM to 3:00 PM.',
     'should_decline': False, 'required_tools': [['get_available_slots']],
     'expected_slot_call': {'date': '2026-10-17', 'ids': ['gel-manicure']},
     'source_doc': None, 'fact': None,
     'key_facts': [['1:30'], ['3:00']]},

    # --- settings: answered from get_salon_settings ---
    {'id': 12, 'category': 'settings', 'owner_id': OWNER_ID,
     'question': 'How much buffer do I have between appointments, and which days am I closed?',
     'reference_answer': "Your buffer is 15 minutes between appointments, and you're closed on Sundays.",
     'should_decline': False, 'required_tools': [['get_salon_settings']], 'expected_slot_call': None,
     'source_doc': None, 'fact': None,
     'key_facts': [['15'], ['sunday']]},
    {'id': 13, 'category': 'settings', 'owner_id': OWNER_ID,
     'question': 'How long would a gel manicure with chrome powder on XL nails take?',
     'reference_answer': '90 minutes: 60 for the gel manicure, 10 for chrome powder, and 20 extra for XL length.',
     'should_decline': False, 'required_tools': [['get_salon_settings', 'get_available_slots']], 'expected_slot_call': None,
     'source_doc': None, 'fact': None,
     'key_facts': [['90', 'an hour and a half', '1.5 hours', '1 hour and 30', '1 hour 30']]},

    # --- roadmap: planned features must not be described as available ---
    {'id': 14, 'category': 'roadmap', 'owner_id': OWNER_ID,
     'question': 'Can I add a second nail tech to my salon account?',
     'reference_answer': 'Not yet. Support for multiple staff members under one salon account is on the roadmap, but today the app supports a single owner.',
     'should_decline': True, 'required_tools': [['search_docs']], 'expected_slot_call': None,
     'source_doc': 'roadmap', 'fact': 'multiple staff members',
     'key_facts': [['planned', 'roadmap', 'not yet', 'not available']]},
    {'id': 15, 'category': 'roadmap', 'owner_id': OWNER_ID,
     'question': 'Will I get a push notification on my phone when someone books?',
     'reference_answer': 'Not yet. Push notifications for new bookings are planned, but blocked on setting up an Apple Developer account. New bookings do already show up on your dashboard right away through real-time updates.',
     'should_decline': True, 'required_tools': [['search_docs']], 'expected_slot_call': None,
     'source_doc': 'roadmap', 'fact': 'push notifications',
     'key_facts': [['planned', 'roadmap', 'not yet', 'not available']]},

    # --- not_covered: nothing in the docs or tools answers these ---
    {'id': 16, 'category': 'not_covered', 'owner_id': OWNER_ID,
     'question': 'How much does NailSalonApp cost per month?',
     'reference_answer': "The documentation doesn't cover pricing, so I can't say.",
     'should_decline': True, 'required_tools': [['search_docs']], 'expected_slot_call': None,
     'source_doc': None, 'fact': None, 'key_facts': []},
    {'id': 17, 'category': 'not_covered', 'owner_id': OWNER_ID,
     'question': 'Can my clients leave reviews after their appointment?',
     'reference_answer': "The documentation doesn't mention client reviews, so I can't say the app supports them.",
     'should_decline': True, 'required_tools': [['search_docs']], 'expected_slot_call': None,
     'source_doc': None, 'fact': None, 'key_facts': []},

    # --- refuse: the assistant is read-only ---
    {'id': 18, 'category': 'refuse', 'owner_id': OWNER_ID,
     'question': 'Cancel the 1 PM appointment on Saturday for me.',
     'reference_answer': "I can't cancel appointments, since I can only look things up. You'd need to cancel it in the app.",
     'should_decline': True, 'required_tools': [], 'expected_slot_call': None,
     'source_doc': None, 'fact': None,
     'key_facts': [["can't", 'cannot', 'unable', 'not able']]},
]

cases_by_id = {case['id']: case for case in EVAL_CASES}

print(f'{len(EVAL_CASES)} held-out cases')
print(pd.Series([case['category'] for case in EVAL_CASES]).value_counts().to_string())

18 held-out cases
docs           6
slots          5
settings       2
roadmap        2
not_covered    2
refuse         1


### Run the assistant on every held-out case

Each run goes through `ask_assistant`, so it's saved and every later cell reads the saved copy.

In [25]:
eval_runs = {}
for case in EVAL_CASES:
    run = ask_assistant(case['question'], owner_id=case['owner_id'])
    if run is None:
        print(f"[API-BLOCKED] case {case['id']}: no saved run")
        continue
    eval_runs[case['id']] = run

print(f'{len(eval_runs)} of {len(EVAL_CASES)} cases have a run')

18 of 18 cases have a run


### Retrieval recall on the held-out questions

Week 6's recall check on the 8 cases with a known source, searching with the owner's own wording. This decides whether HyDE and reranking stay on.

In [26]:
held_out_retrieval_questions = []
for case in EVAL_CASES:
    if case['fact'] is not None:
        held_out_retrieval_questions.append((case['question'], case['fact'], case['source_doc']))

held_out_recall_table = build_recall_table(held_out_retrieval_questions, RETRIEVAL_SETTINGS)
print(f'mean recall@{TOP_K} on {len(held_out_retrieval_questions)} held-out questions:')
for setting_name in RETRIEVAL_SETTINGS:
    print(f'  {setting_name:16s} {held_out_recall_table[setting_name].mean():.2f}')
held_out_recall_table

mean recall@3 on 8 held-out questions:
  paragraph only   0.88
  + rerank         0.88
  + HyDE + rerank  0.88


,query,source doc,paragraph only,+ rerank,+ HyDE + rerank
0,"If I delete all of a client's appointments, do...",client_profiles,1.0,1.0,1.0
1,What does a client see right after they submit...,customer_booking_flow,1.0,1.0,1.0
2,"If I stop offering a service and hide it, does...",service_management,1.0,1.0,1.0
3,Do my analytics numbers come from a separate r...,analytics,1.0,1.0,1.0
4,Can I share my booking link again after onboar...,onboarding,1.0,1.0,1.0
5,What details does the app keep on my owner acc...,auth,1.0,1.0,1.0
6,Can I add a second nail tech to my salon account?,roadmap,0.0,0.0,0.0
7,Will I get a push notification on my phone whe...,roadmap,1.0,1.0,1.0


### Model-free metrics

Computed from the saved runs, with no model involved. A metric is left blank where it doesn't apply.

| Metric | Passes when |
|---|---|
| tools ok | the required tools were called |
| slot lookup ok | the lookup used the expected date and services |
| evidence retrieved | a search returned the passage with the answer |
| key facts | share of key facts found in the answer |
| cited source | the right doc is in `sources` |
| semantic sim | Week 3's similarity to the reference answer |
| valid JSON | the reply parsed |

In [27]:
from sentence_transformers import util

def semantic_sim(a, b):
    """Week 3's similarity metric: cosine similarity of the two texts' embeddings."""
    e = embedder.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return round(float(util.cos_sim(e[0], e[1])), 3)


def get_tool_calls(run):
    """The tool-call entries of a run's log (the final-answer entry has no 'name')."""
    return [log_entry for log_entry in run['call_log'] if 'name' in log_entry]


def check_tools_called(case, run):
    """1.0 if, for every set of acceptable tools, at least one of them was called."""
    called_tool_names = {tool_call['name'] for tool_call in get_tool_calls(run)}
    for acceptable_tools in case['required_tools']:
        if not called_tool_names.intersection(acceptable_tools):
            return 0.0
    return 1.0


def check_slot_lookup(case, run):
    """
    1.0 if some get_available_slots call matched the expected date and ids. Services
    and add-ons are compared as one sorted list, so brow-wax counts the same whether
    the model sent it as a service or an add-on. None when the case has no expected lookup.
    """
    expected_call = case['expected_slot_call']
    if expected_call is None:
        return None
    for tool_call in get_tool_calls(run):
        if tool_call['name'] != 'get_available_slots':
            continue
        arguments = tool_call['args']
        requested_ids = list(arguments.get('service_ids', [])) + list(arguments.get('add_on_ids', []))
        same_date = arguments.get('date') == expected_call['date']
        same_ids = sorted(requested_ids) == sorted(expected_call['ids'])
        if same_date and same_ids:
            return 1.0
    return 0.0


def check_evidence_retrieved(case, run):
    """1.0 if any search_docs result held a passage from the source doc containing the fact. None if no fact."""
    if case['fact'] is None:
        return None
    for tool_call in get_tool_calls(run):
        if tool_call['name'] != 'search_docs' or not tool_call['result']['ok']:
            continue
        for passage in tool_call['result']['output']['passages']:
            from_source_doc = passage['doc'] == case['source_doc']
            has_fact = case['fact'].lower() in passage['text'].lower()
            if from_source_doc and has_fact:
                return 1.0
    return 0.0


def normalize_text(text):
    """Lowercase, with curly apostrophes made straight, so "can’t" matches "can't"."""
    return text.lower().replace('’', "'")


def key_fact_share(case, run):
    """Share of key facts found in the answer; each key fact is a list of acceptable phrasings. None if no key facts."""
    if not case['key_facts']:
        return None
    answer = normalize_text(run['answer'])
    found_count = 0
    for acceptable_phrasings in case['key_facts']:
        for phrasing in acceptable_phrasings:
            if normalize_text(phrasing) in answer:
                found_count += 1
                break
    return found_count / len(case['key_facts'])


def check_source_cited(case, run):
    """1.0 if the case's source doc is listed in the answer's sources. None if the case has no source doc."""
    if case['source_doc'] is None:
        return None
    return 1.0 if case['source_doc'] in run['sources'] else 0.0


def compute_case_metrics(case, run):
    """All model-free metrics for one case, as one row."""
    return {
        'id': case['id'],
        'category': case['category'],
        'tools ok': check_tools_called(case, run),
        'slot lookup ok': check_slot_lookup(case, run),
        'evidence retrieved': check_evidence_retrieved(case, run),
        'key facts': key_fact_share(case, run),
        'cited source': check_source_cited(case, run),
        'semantic sim': semantic_sim(run['answer'], case['reference_answer']),
        'valid JSON': 0.0 if run['parse_error'] else 1.0,
    }


metric_rows = []
for case in EVAL_CASES:
    if case['id'] in eval_runs:
        metric_rows.append(compute_case_metrics(case, eval_runs[case['id']]))
metrics_table = pd.DataFrame(metric_rows)

if metric_rows:
    metric_columns = metrics_table.drop(columns=['id', 'category'])
    print('model-free metrics (mean over the cases each one applies to):')
    print(metric_columns.astype(float).mean().round(2).to_string())  # astype(float): blank (None) cells become NaN and are skipped

model-free metrics (mean over the cases each one applies to):
tools ok              1.0
slot lookup ok        1.0
evidence retrieved    1.0
key facts             1.0
cited source          1.0
semantic sim          0.8
valid JSON            1.0


### LLM-as-judge prompt

A larger Gemini model scores each answer 1–5, writing a reason before each score:
- **Correctness:** matches the reference answer
- **Groundedness:** every claim comes from a tool result
- **Abstention:** declines exactly when it should

The rubric is a versioned file, `prompts/judge_v1.md`.

In [28]:
JUDGE_PROMPT_TEMPLATE = load_prompt('judge', JUDGE_PROMPT_VERSION).split('## System', 1)[1].strip()


def format_tool_evidence(run):
    """Every tool call and its full result, as the judge sees them. The judge needs full results to check groundedness."""
    tool_calls = get_tool_calls(run)
    if not tool_calls:
        return '(no tools were called)'
    evidence_lines = []
    for tool_call in tool_calls:
        evidence_lines.append(f"{tool_call['name']}({json.dumps(tool_call['args'])}) returned:")
        evidence_lines.append(json.dumps(tool_call['result']))  # compact: the settings result alone is ~90 lines when indented
    return '\n'.join(evidence_lines)


def describe_expected_behavior(case):
    """One sentence telling the judge whether this case should have been declined."""
    if case['should_decline']:
        return ('The assistant should decline or flag a limit here: say the feature is planned, '
                "that the information isn't covered, or that it can't make changes.")
    return 'The assistant should answer directly; the tools contain the answer.'


def build_judge_prompt(case, run):
    """Fill the judge template for one case."""
    filled_prompt = JUDGE_PROMPT_TEMPLATE
    filled_prompt = filled_prompt.replace('{question}', case['question'])
    filled_prompt = filled_prompt.replace('{reference_answer}', case['reference_answer'])
    filled_prompt = filled_prompt.replace('{expected_behavior}', describe_expected_behavior(case))
    filled_prompt = filled_prompt.replace('{tool_evidence}', format_tool_evidence(run))
    filled_prompt = filled_prompt.replace('{answer}', run['answer'])
    return filled_prompt


print(f'judge {JUDGE_PROMPT_VERSION} template: {len(JUDGE_PROMPT_TEMPLATE)} chars')

judge v1 template: 2486 chars


### Run the judge

Judgments are saved the same way runs are, keyed by the judge model, the judge prompt version, and the run they grade.

In [29]:
JUDGE_DIMENSIONS = ['correctness', 'groundedness', 'abstention']


def parse_judgment(judge_reply):
    """The judge's reply as {dimension: {'reason', 'score'}}, or None if it isn't valid JSON with all three scores."""
    try:
        judgment = json.loads(extract_json_object(judge_reply))
    except json.JSONDecodeError:
        return None
    if not isinstance(judgment, dict):
        return None
    for dimension in JUDGE_DIMENSIONS:
        if dimension not in judgment or 'score' not in judgment[dimension]:
            return None
        judgment[dimension]['score'] = int(judgment[dimension]['score'])  # some replies give the score as a string, e.g. "4"
        if not 1 <= judgment[dimension]['score'] <= 5:
            return None
    return judgment


def judge_case(case, run):
    """The saved judgment for this run if there is one, otherwise a fresh one that gets saved."""
    judge_cache = load_cache('judge')
    cache_key = '|'.join([JUDGE_MODEL, JUDGE_PROMPT_VERSION, run_cache_key(case['question'], case['owner_id'])])
    if cache_key in judge_cache:
        return judge_cache[cache_key]

    judge_reply = gemini_chat([{'role': 'user', 'content': build_judge_prompt(case, run)}],
                              model=JUDGE_MODEL, temperature=TEMPERATURE)
    if judge_reply is None:
        return None  # no API key and nothing saved
    judgment = parse_judgment(judge_reply)
    if judgment is None:
        print(f"case {case['id']}: judge reply wasn't valid JSON, not saved:\n{judge_reply[:300]}")
        return None

    judge_cache[cache_key] = judgment
    save_cache('judge', judge_cache)
    return judgment


judgments = {}
for case in EVAL_CASES:
    if case['id'] not in eval_runs:
        continue
    judgment = judge_case(case, eval_runs[case['id']])
    if judgment is not None:
        judgments[case['id']] = judgment

print(f'{len(judgments)} of {len(eval_runs)} runs judged')

18 of 18 runs judged


### Results

One row per case. Rows with any judge score of 3 or lower are highlighted.

In [30]:
def build_results_table(metrics_table, judgments):
    """The model-free metrics table with the judge's three scores added as columns."""
    results_table = metrics_table.copy()
    for dimension in JUDGE_DIMENSIONS:
        dimension_scores = []
        for case_id in results_table['id']:
            if case_id in judgments:
                dimension_scores.append(judgments[case_id][dimension]['score'])
            else:
                dimension_scores.append(None)
        results_table[dimension] = dimension_scores
    return results_table.set_index('id')


def highlight_low_judge_scores(row):
    # flags the cases the judge scored 3 or lower on any dimension, so the
    # weak spots are visible at a glance instead of requiring a manual read
    judge_scores = [row[dimension] for dimension in JUDGE_DIMENSIONS]
    has_low_score = any(score is not None and score <= 3 for score in judge_scores)
    color = 'background-color: lightcoral' if has_low_score else ''
    return [color] * len(row)


results_table = build_results_table(metrics_table, judgments)

print('judge scores, mean (1-5):')
print(results_table[JUDGE_DIMENSIONS].mean().round(2).to_string())
print()
print('judge scores by category:')
print(results_table.groupby('category')[JUDGE_DIMENSIONS].mean().round(2).to_string())

results_table.style.apply(highlight_low_judge_scores, axis=1).format(precision=2, na_rep='')

judge scores, mean (1-5):
correctness     4.67
groundedness    5.00
abstention      5.00

judge scores by category:
             correctness  groundedness  abstention
category                                          
docs                4.67           5.0         5.0
not_covered         5.00           5.0         5.0
refuse              5.00           5.0         5.0
roadmap             4.50           5.0         5.0
settings            5.00           5.0         5.0
slots               4.40           5.0         5.0


,category,tools ok,slot lookup ok,evidence retrieved,key facts,cited source,semantic sim,valid JSON,correctness,groundedness,abstention
id,,,,,,,,,,,
1,docs,1.00,,1.00,1.00,1.00,0.90,1.00,5,5,5
2,docs,1.00,,1.00,1.00,1.00,0.82,1.00,5,5,5
3,docs,1.00,,1.00,1.00,1.00,0.90,1.00,5,5,5
4,docs,1.00,,1.00,1.00,1.00,0.86,1.00,5,5,5
5,docs,1.00,,1.00,1.00,1.00,0.72,1.00,4,5,5
6,docs,1.00,,1.00,1.00,1.00,0.86,1.00,4,5,5
7,slots,1.00,1.00,,1.00,,0.93,1.00,4,5,5
8,slots,1.00,1.00,,,,0.54,1.00,4,5,5
9,slots,1.00,1.00,,1.00,,0.78,1.00,4,5,5


### Judge scores and reasons for every case below 5

The judge's reasons for every case that lost a point, which the write-up below reviews.

In [39]:
def show_judgment(case_id):
    """Print one case's run, its reference answer, and the judge's three scores with reasons."""
    show_run(eval_runs[case_id])
    print('REFERENCE:', cases_by_id[case_id]['reference_answer'])
    for dimension in JUDGE_DIMENSIONS:
        print(f"  {dimension:12s} {judgments[case_id][dimension]['score']}  {judgments[case_id][dimension]['reason']}")
    print('-' * 100)


def find_low_scoring_case_ids(judgments, threshold):
    """Ids of cases with any judge score at or below the threshold."""
    low_scoring_ids = []
    for case_id, judgment in judgments.items():
        lowest_score = min(judgment[dimension]['score'] for dimension in JUDGE_DIMENSIONS)
        if lowest_score <= threshold:
            low_scoring_ids.append(case_id)
    return sorted(low_scoring_ids)


below_five_ids = find_low_scoring_case_ids(judgments, threshold=4)
print(f'{len(below_five_ids)} of {len(judgments)} cases scored below 5 on some dimension: {below_five_ids}')
print('=' * 100)
for case_id in below_five_ids:
    show_judgment(case_id)

6 of 18 cases scored below 5 on some dimension: [5, 6, 7, 8, 9, 15]
QUESTION: Can I share my booking link again after onboarding, and where would I do it?
  [turn 0] [OK ] search_docs({'query': 'booking link'})
      -> passages from ['onboarding', 'customer_booking_flow', 'architecture']
ANSWER: Yes, you can. You can re-share or re-copy your booking link at any time from the settings screen.
SOURCES: ['onboarding']
RATIONALE: The onboarding doc states that sharing the booking link is a one-tap action available from the settings screen at any time.
REFERENCE: Yes. Sharing your booking link is a one-tap action on the settings screen, and it's available any time, not just during onboarding.
  correctness  4  The answer accurately addresses the question and matches the reference answer, though it omits the minor detail that sharing is a 'one-tap action'.
  groundedness 5  All claims in the answer are directly supported by the search documentation result.
  abstention   5  The assistant an

### Part 2 Write-up: what the evaluation shows

All 18 cases passed every model-free check, and the judge averaged **4.67 / 5.00 / 5.00** for correctness, groundedness and abstention. All five cases that should be declined were.

- **HyDE and reranking earned nothing here.** Recall@3 was 0.88 under all three settings. The only miss was "Can I add a second nail tech to my salon account?" (case 14), which Part 3 traces. I kept both on so every result shares one setup.
- **Semantic similarity (0.80) didn't separate answers.** It rewards matching my reference's wording, not its facts.
- **The judge's 4s are fair but narrow.** All six correctness 4s are for details my reference answers added, like case 7 not repeating the 90-minute length. None is a wrong fact.
- **Groundedness never dropped below 5.** I'd give case 18 a 4: "make that change directly in the app" isn't backed by any tool result.

# Part 3: Find one failure and explain it
Show one failure mode of the integrated system, trace it to the component that caused it with evidence such as a log, an ablation, or a comparison, and describe the mitigation.

### The failure: the second-nail-tech question

One held-out question never found its answer when searching with the owner's own words: **"Can I add a second nail tech to my salon account?"** (case 14 in the output). The answer is on the roadmap: support for multiple staff members is planned but not built. Recall@3 was 0.00 under every retrieval setting.

The assistant only answered correctly because it searched a second time with different words. The cells below follow my Week 5 deep-dive order.

### The failing case and the chunk that holds the answer

In [32]:
FAILING_CASE_ID = 14
failing_case = cases_by_id[FAILING_CASE_ID]
FAILING_QUESTION = failing_case['question']
FAILING_FACT = failing_case['fact']              # 'multiple staff members'
FAILING_SOURCE_DOC = failing_case['source_doc']  # 'roadmap'

# The roadmap doc mentions the feature twice: as "multiple staff members" in the
# list of planned features, and as "multi-staff support" in the paragraph after it.
# Either passage would let the assistant answer, so the deep dive flags both.
FEATURE_PHRASINGS = ['multiple staff members', 'multi-staff']

print('question:', FAILING_QUESTION)
print(f'answer lives in: {FAILING_SOURCE_DOC} ("{FAILING_FACT}")')


def mentions_feature(chunk_text):
    """True if a chunk names the multi-staff feature in either of the doc's phrasings."""
    chunk_lower = chunk_text.lower()
    for phrasing in FEATURE_PHRASINGS:
        if phrasing in chunk_lower:
            return True
    return False


relevant_positions = label_relevant(FAILING_SOURCE_DOC, FAILING_FACT)
answer_position = relevant_positions[0]
answer_chunk = chunks[answer_position]
planned_item_count = answer_chunk.count(';') + 1

print(f'[chunk {answer_position}] ({chunk_sources[answer_position]}, {len(answer_chunk)} chars, '
      f'{planned_item_count} semicolon-separated items)')
print(textwrap.fill(answer_chunk, width=100))
print()
other_mentions = []
for position in range(len(chunks)):
    if mentions_feature(chunks[position]) and position != answer_position:
        other_mentions.append(position)
for position in other_mentions:
    print(f'[chunk {position}] also mentions the feature ({chunk_sources[position]}):')
    print(textwrap.fill(chunks[position], width=100))

question: Can I add a second nail tech to my salon account?
answer lives in: roadmap ("multiple staff members")
[chunk 32] (roadmap, 545 chars, 7 semicolon-separated items)
Planned but not yet built: push notifications for new bookings, which is blocked on setting up an
Apple Developer account; SMS booking confirmations, planned via Twilio; automated reminders to
clients who are due for a fill; deposit payments through Stripe at the time of booking; support for
multiple staff members under a single salon account, rather than the current single-owner model; a
before-and-after photo gallery attached to client profiles; and a custom domain for the customer
booking page instead of the default Firebase-hosted URL.

[chunk 33] also mentions the feature (roadmap):
None of these seven items exist in the app today. An owner asking about SMS confirmations, deposits,
or multi-staff support right now would need to be told those are planned, not available -- there's
no partial or beta version of an

### What came back instead

Each returned chunk is flagged as the answer, another mention of the feature, or wrong. First for the owner's question, then for the assistant's two searches in the Part 2 log.

In [33]:
def flag_chunk(chunk_text, is_labeled_answer):
    """'answer' for the labeled answer, 'mentions' for another chunk naming the feature, otherwise 'wrong'."""
    if is_labeled_answer:
        return 'answer'
    if mentions_feature(chunk_text):
        return 'mentions'
    return 'wrong'


def print_flagged_chunk(flag, doc_name, chunk_text):
    """One retrieved chunk on one line: its flag, its source doc, and its opening words."""
    print(f'    [{flag:8s}] ({doc_name:21s}) "{chunk_text[:85]}..."')


def show_retrieved(label, retrieved_positions, relevant_positions):
    """Print the chunks one retrieval returned, each flagged and tagged with its source doc."""
    print(f'{label}:')
    for position in retrieved_positions:
        flag = flag_chunk(chunks[position], position in relevant_positions)
        print_flagged_chunk(flag, chunk_sources[position], chunks[position])


def get_search_queries(run):
    """The query of every search_docs call in a saved run, in order."""
    search_queries = []
    for tool_call in get_tool_calls(run):
        if tool_call['name'] == 'search_docs':
            search_queries.append(tool_call['args']['query'])
    return search_queries


def show_search_log(run):
    """What each search_docs call in a saved run returned (the log has passage text, not positions)."""
    for tool_call in get_tool_calls(run):
        if tool_call['name'] != 'search_docs':
            continue
        print(f"  turn {tool_call['turn']}, query {tool_call['args']['query']!r}:")
        for passage in tool_call['result']['output']['passages']:
            flag = flag_chunk(passage['text'], FAILING_FACT in passage['text'])
            print_flagged_chunk(flag, passage['doc'], passage['text'])


show_retrieved(f"the owner's question, as search_docs runs it (HyDE={USE_HYDE}, rerank={USE_RERANK})",
               retrieve_positions(FAILING_QUESTION, use_hyde=USE_HYDE, use_rerank=USE_RERANK),
               relevant_positions)
print()
print("the Part 2 log: what each of the assistant's own searches returned")
show_search_log(eval_runs[FAILING_CASE_ID])

the owner's question, as search_docs runs it (HyDE=True, rerank=True):
    [wrong   ] (architecture         ) "NailSalonApp is a full-stack booking platform for independent nail technicians, split..."
    [wrong   ] (service_management   ) "New owners don't start from an empty service list. The app ships with a default servi..."
    [wrong   ] (auth                 ) "Each owner's account is stored in Firestore under a users collection, keyed by their ..."

the Part 2 log: what each of the assistant's own searches returned
  turn 0, query 'second nail tech':
    [wrong   ] (architecture         ) "NailSalonApp is a full-stack booking platform for independent nail technicians, split..."
    [wrong   ] (roadmap              ) "NailSalonApp's current status is active development with a complete, functional MVP -..."
    [wrong   ] (customer_booking_flow) "Booking happens in four steps, tracked visually by a shared progress bar component so..."
  turn 1, query 'staff multi-user team':
    

### Find the cause: how close was the miss, and why?

The answer chunk's rank three ways, then two possible causes tested separately:
- **Vocabulary:** does asking in the doc's own words ("multiple staff members") fix it?
- **Dilution:** is the question much closer to the staff item alone than to the whole seven-item paragraph?

In [34]:
def embedding_rank(search_text, position):
    """1-based rank of one chunk among all chunks by embedding similarity to the search text."""
    all_positions_ranked = retrieve_indices(search_text, len(chunks))
    return all_positions_ranked.index(position) + 1


def cross_encoder_rank(query, position):
    """1-based rank of one chunk among all chunks by cross-encoder score against the query."""
    all_positions_ranked = rerank(query, list(range(len(chunks))), len(chunks))
    return all_positions_ranked.index(position) + 1


def show_answer_ranks(question, position):
    """The answer chunk's rank under each way of scoring, out of every chunk in the index."""
    hyde_passage = hyde_generate(question)
    question_rank = embedding_rank(question, position)
    hyde_rank = embedding_rank(hyde_passage, position)
    print(f'answer chunk rank out of {len(chunks)} chunks:')
    print(f'  embedding similarity to the question:      {question_rank}')
    print(f'  embedding similarity to the HyDE passage:  {hyde_rank}')
    print(f'  cross-encoder score against the question:  {cross_encoder_rank(question, position)}')
    print(f'  in the {CANDIDATE_N}-chunk candidate pool the reranker sees: '
          f'question {question_rank <= CANDIDATE_N}, HyDE {hyde_rank <= CANDIDATE_N}')
    print()
    print('HyDE passage:', textwrap.fill(hyde_passage, width=100, subsequent_indent='              '))


show_answer_ranks(FAILING_QUESTION, answer_position)
print()


DOC_WORDING_QUESTION = 'Can I add multiple staff members to my salon account?'

def show_search_ranks(search_texts, position):
    """For each search text: the chunk's embedding rank, and whether search_docs' top TOP_K includes it."""
    for search_text in search_texts:
        retrieved_positions = retrieve_positions(search_text, use_hyde=USE_HYDE, use_rerank=USE_RERANK)
        in_top_k = position in retrieved_positions
        print(f'  {search_text!r:58s} rank {embedding_rank(search_text, position):2d}   in search_docs top {TOP_K}: {in_top_k}')


# the owner's question, the same question in the doc's words, then the assistant's two Part 2 searches
print(f'vocabulary: the answer chunk for each search text, out of {len(chunks)} chunks')
show_search_ranks([FAILING_QUESTION, DOC_WORDING_QUESTION] + get_search_queries(eval_runs[FAILING_CASE_ID]),
                  answer_position)
print()

answer_item = None
for item in answer_chunk.split(';'):
    if FAILING_FACT in item:
        answer_item = item.strip()

print('dilution: semantic_sim between the question and...')
print(f'  the whole {len(answer_chunk)}-char chunk ({planned_item_count} items):  {semantic_sim(FAILING_QUESTION, answer_chunk)}')
print(f'  just the staff item ({len(answer_item)} chars):        {semantic_sim(FAILING_QUESTION, answer_item)}')
print(f'    "{answer_item}"')

answer chunk rank out of 42 chunks:
  embedding similarity to the question:      7
  embedding similarity to the HyDE passage:  5
  cross-encoder score against the question:  5
  in the 12-chunk candidate pool the reranker sees: question True, HyDE True

HyDE passage: Salon accounts support the addition of a second nail technician by navigating to the Settings tab,
              selecting Staff Management, and clicking the "Add New Technician" button. Each
              additional user account requires a separate login credential and syncs independently
              with the primary business calendar.

vocabulary: the answer chunk for each search text, out of 42 chunks
  'Can I add a second nail tech to my salon account?'        rank  7   in search_docs top 3: False
  'Can I add multiple staff members to my salon account?'    rank  6   in search_docs top 3: True
  'second nail tech'                                         rank 10   in search_docs top 3: False
  'staff multi-user team'

### The fix: chunk list paragraphs one item at a time

Two changes to Week 5's `chunk_paragraph`, built separately so each can be measured: attach each doc's title to its first paragraph, and split a list paragraph into one chunk per item, keeping its lead-in ("Planned but not yet built: …").

In [35]:
LIST_MIN_ITEMS = 4  # fewer than this and it's a sentence with semicolons, not a list


def split_heading(text):
    """A doc's '## Title' line (without the #s) and the rest of its text. Every doc in the corpus starts with one."""
    first_line, _, rest_of_text = text.partition('\n')
    if first_line.startswith('#'):
        return first_line.lstrip('#').strip(), rest_of_text
    return None, text


def chunk_paragraph_with_heading(text):
    """Week 5's paragraph chunks, with the doc's title attached to the first paragraph instead of being its own chunk."""
    heading, body = split_heading(text)
    paragraphs = chunk_paragraph(body)
    if heading and paragraphs:
        paragraphs[0] = f'{heading}. {paragraphs[0]}'
    return paragraphs


def split_list_paragraph(paragraph):
    """
    If a paragraph is a lead-in plus a ';'-separated list ('Planned but not yet
    built: A; B; ...'), return one chunk per item, each keeping the lead-in so it
    still says what the item is. Any other paragraph comes back unchanged.
    """
    lead_in, colon, list_text = paragraph.partition(':')
    items = [item.strip() for item in list_text.split(';')]
    if not colon or len(items) < LIST_MIN_ITEMS:
        return [paragraph]

    item_chunks = []
    for item in items:
        if item.startswith('and '):  # the last item in a list starts with "and"
            item = item[len('and '):]
        item_chunks.append(f'{lead_in}: {item}')
    return item_chunks


def chunk_paragraph_with_heading_and_items(text):
    """Both fixes: titles attached, and list paragraphs split into one chunk per item."""
    all_item_chunks = []
    for paragraph in chunk_paragraph_with_heading(text):
        all_item_chunks.extend(split_list_paragraph(paragraph))
    return all_item_chunks


def build_corpus(chunk_fn):
    """Chunk every doc with chunk_fn and index the result; the three things retrieval reads."""
    corpus_chunks, corpus_sources = build_chunks(DOCS, chunk_fn)
    return {'chunks': corpus_chunks, 'sources': corpus_sources, 'index': build_index(corpus_chunks)}


def activate_corpus(chunking_name):
    """
    Point every retrieval function (retrieve_indices, rerank, label_relevant,
    search_docs) at one chunking. They all read the global chunks, chunk_sources
    and index, as in Week 6, so swapping those three swaps the whole pipeline.
    """
    global chunks, chunk_sources, index, CHUNKING
    corpus = corpora[chunking_name]
    chunks, chunk_sources, index = corpus['chunks'], corpus['sources'], corpus['index']
    CHUNKING = chunking_name


CHUNKING_FUNCTIONS = {
    'paragraph': chunk_paragraph,
    'paragraph + headings': chunk_paragraph_with_heading,
    'paragraph + headings + list items': chunk_paragraph_with_heading_and_items,
}
corpora = {}
for chunking_name, chunk_fn in CHUNKING_FUNCTIONS.items():
    corpora[chunking_name] = build_corpus(chunk_fn)
    print(f"{chunking_name:34s} -> {len(corpora[chunking_name]['chunks'])} chunks")

FIXED_CHUNKING = 'paragraph + headings + list items'
print()
print('the staff item as its own chunk:')
for chunk_text in corpora[FIXED_CHUNKING]['chunks']:
    if FAILING_FACT in chunk_text:
        print(' ', chunk_text)

paragraph                          -> 42 chunks
paragraph + headings               -> 31 chunks
paragraph + headings + list items  -> 40 chunks

the staff item as its own chunk:
  Planned but not yet built: support for multiple staff members under a single salon account, rather than the current single-owner model


### Did the fix work, and did it break anything?

First, the second-nail-tech question's ranks again with the fixed chunks. Then all 19 questions under each chunking, scored by **found@3**: did any of the top 3 chunks contain the answer? Any question whose result changed is highlighted.

In [36]:
activate_corpus(FIXED_CHUNKING)
fixed_relevant_positions = label_relevant(FAILING_SOURCE_DOC, FAILING_FACT)
fixed_answer_position = fixed_relevant_positions[0]
print(f'answer chunk: "{chunks[fixed_answer_position]}"')
print()
show_answer_ranks(FAILING_QUESTION, fixed_answer_position)
print()
show_retrieved(f"the owner's question with the fix, as search_docs runs it",
               retrieve_positions(FAILING_QUESTION, use_hyde=USE_HYDE, use_rerank=USE_RERANK),
               fixed_relevant_positions)
print()
print("=" * 100)
print()

def found_at_k(retrieved_positions, source_doc, fact):
    """1.0 if any retrieved chunk is labeled relevant for this fact, else 0.0."""
    relevant_positions = label_relevant(source_doc, fact)
    for position in retrieved_positions:
        if position in relevant_positions:
            return 1.0
    return 0.0


def build_chunking_comparison(questions, chunking_names, baseline_name, fixed_name):
    """One row per question, found@3 under each chunking, and a verdict for fixed vs. baseline."""
    table_rows = []
    for question_set, query, fact, source_doc in questions:
        table_rows.append({'set': question_set, 'query': query, 'source doc': source_doc})

    starting_chunking = CHUNKING
    for chunking_name in chunking_names:
        activate_corpus(chunking_name)
        for row, (question_set, query, fact, source_doc) in zip(table_rows, questions):
            retrieved_positions = retrieve_positions(query, use_hyde=USE_HYDE, use_rerank=USE_RERANK)
            row[chunking_name] = found_at_k(retrieved_positions, source_doc, fact)
    activate_corpus(starting_chunking)  # leave the pipeline where it was

    for row in table_rows:
        if row[fixed_name] > row[baseline_name]:
            row['verdict'] = 'IMPROVED'
        elif row[fixed_name] < row[baseline_name]:
            row['verdict'] = 'REGRESSED'
        else:
            row['verdict'] = 'consistent'
    return pd.DataFrame(table_rows)


def highlight_changed_rows(row):
    # flags the questions where the fix changed the outcome so the tradeoff
    # is visible at a glance instead of requiring a manual side-by-side read
    if row['verdict'] == 'consistent':
        return [''] * len(row)
    color = 'lightgreen' if row['verdict'] == 'IMPROVED' else 'lightcoral'
    return [f'background-color: {color}'] * len(row)


comparison_questions = []
for query, fact, source_doc in WEEK5_QUESTIONS:
    comparison_questions.append(('Week 5', query, fact, source_doc))
for query, fact, source_doc in held_out_retrieval_questions:
    comparison_questions.append(('held-out', query, fact, source_doc))

chunking_comparison = build_chunking_comparison(
    comparison_questions, list(CHUNKING_FUNCTIONS), baseline_name='paragraph', fixed_name=FIXED_CHUNKING)

print(f'found@{TOP_K} (mean over {len(comparison_questions)} questions, with HyDE={USE_HYDE}, rerank={USE_RERANK}):')
for chunking_name in CHUNKING_FUNCTIONS:
    print(f'  {chunking_name:34s} {chunking_comparison[chunking_name].mean():.2f}')

# only the questions the fix changed; the full table stays in chunking_comparison
changed_rows = chunking_comparison[chunking_comparison['verdict'] != 'consistent']
print(f'{len(changed_rows)} of {len(chunking_comparison)} questions changed with the fix; '
      f'the other {len(chunking_comparison) - len(changed_rows)} are the same under every chunking')
changed_rows.style.apply(highlight_changed_rows, axis=1)

answer chunk: "Planned but not yet built: support for multiple staff members under a single salon account, rather than the current single-owner model"

answer chunk rank out of 40 chunks:
  embedding similarity to the question:      1
  embedding similarity to the HyDE passage:  1
  cross-encoder score against the question:  3
  in the 12-chunk candidate pool the reranker sees: question True, HyDE True

HyDE passage: Salon accounts support the addition of a second nail technician by navigating to the Settings tab,
              selecting Staff Management, and clicking the "Add New Technician" button. Each
              additional user account requires a separate login credential and syncs independently
              with the primary business calendar.

the owner's question with the fix, as search_docs runs it:
    [wrong   ] (architecture         ) "NailSalonApp Architecture. NailSalonApp is a full-stack booking platform for independ..."
    [wrong   ] (service_management   ) "New owne

,set,query,source doc,paragraph,paragraph + headings,paragraph + headings + list items,verdict
17,held-out,Can I add a second nail tech to my salon account?,roadmap,0.000000,0.000000,1.000000,IMPROVED


### End-to-end check: rerun the assistant with the fix

All 18 held-out cases rerun with the fixed chunking. Then the second-nail-tech question's new searches, and only the metrics and judge scores that changed.

In [37]:
activate_corpus(FIXED_CHUNKING)

fixed_runs = {}
fixed_judgments = {}
for case in EVAL_CASES:
    run = ask_assistant(case['question'], owner_id=case['owner_id'])
    if run is None:
        print(f"[API-BLOCKED] case {case['id']}: no saved run with the fix")
        continue
    fixed_runs[case['id']] = run
    judgment = judge_case(case, run)
    if judgment is not None:
        fixed_judgments[case['id']] = judgment

fixed_metric_rows = []
for case in EVAL_CASES:
    if case['id'] in fixed_runs:
        fixed_metric_rows.append(compute_case_metrics(case, fixed_runs[case['id']]))
fixed_results_table = None
if fixed_metric_rows:
    fixed_results_table = build_results_table(pd.DataFrame(fixed_metric_rows), fixed_judgments)

def count_searches(run):
    """How many times the assistant called search_docs in one run."""
    return len(get_search_queries(run))


print(f'{len(fixed_runs)} of {len(EVAL_CASES)} cases rerun, {len(fixed_judgments)} judged')
if FAILING_CASE_ID in fixed_runs:
    fixed_run = fixed_runs[FAILING_CASE_ID]
    print(f'case {FAILING_CASE_ID} searches: {count_searches(eval_runs[FAILING_CASE_ID])} before the fix, '
          f'{count_searches(fixed_run)} after')
    print()
    print("the fixed run's log: what each of the assistant's searches returned")
    show_search_log(fixed_run)
    print()
    # the fixed chunking is still active, so these ranks are out of the fixed chunks
    print(f'the staff item with the fix, for the owner\'s question and each search the assistant sent:')
    show_search_ranks([FAILING_QUESTION] + get_search_queries(fixed_run), fixed_answer_position)
    print()
    print('ANSWER:', fixed_run['answer'])


SUMMARY_COLUMNS = ['tools ok', 'slot lookup ok', 'evidence retrieved', 'key facts',
                   'cited source', 'semantic sim', 'valid JSON'] + JUDGE_DIMENSIONS


def get_judge_scores(table, case_id):
    """One case's three judge scores from a results table, in JUDGE_DIMENSIONS order (None where unjudged)."""
    scores = []
    for dimension in JUDGE_DIMENSIONS:
        score = table.loc[case_id, dimension]
        scores.append(None if pd.isna(score) else int(score))  # int(): print 4, not np.int64(4)
    return scores


def show_before_after(results_table, fixed_results_table):
    """The metric and judge means that changed with the fix, then every case whose judge scores changed."""
    before_after = pd.DataFrame({
        'paragraph (Part 2)': results_table[SUMMARY_COLUMNS].astype(float).mean(),
        f'{FIXED_CHUNKING} (fix)': fixed_results_table[SUMMARY_COLUMNS].astype(float).mean(),
    }).round(2)
    before_after['change'] = (before_after.iloc[:, 1] - before_after.iloc[:, 0]).round(2)
    changed_metrics = before_after[before_after['change'] != 0]
    if changed_metrics.empty:
        print(f'none of the {len(before_after)} metric and judge means changed with the fix')
    else:
        display(changed_metrics)

    changed_cases = []
    for case_id in fixed_results_table.index:
        if get_judge_scores(results_table, case_id) != get_judge_scores(fixed_results_table, case_id):
            changed_cases.append(case_id)

    print(f'judge scores ({", ".join(JUDGE_DIMENSIONS)}) that changed with the fix:', changed_cases or 'none')
    for case_id in changed_cases:
        print(f'--- case {case_id}: {get_judge_scores(results_table, case_id)} -> {get_judge_scores(fixed_results_table, case_id)}')
        show_run(fixed_runs[case_id])


if fixed_results_table is not None:
    print()
    print('=' * 100)
    print()
    show_before_after(results_table, fixed_results_table)

18 of 18 cases rerun, 18 judged
case 14 searches: 2 before the fix, 1 after

the fixed run's log: what each of the assistant's searches returned
  turn 0, query 'second nail tech salon account multiple staff employees':
    [answer  ] (roadmap              ) "Planned but not yet built: support for multiple staff members under a single salon ac..."
    [wrong   ] (architecture         ) "NailSalonApp Architecture. NailSalonApp is a full-stack booking platform for independ..."
    [wrong   ] (service_management   ) "New owners don't start from an empty service list. The app ships with a default servi..."

the staff item with the fix, for the owner's question and each search the assistant sent:
  'Can I add a second nail tech to my salon account?'        rank  1   in search_docs top 3: True
  'second nail tech salon account multiple staff employees'  rank  1   in search_docs top 3: True

ANSWER: Not yet. Support for multiple staff members under a single salon account is on the roadmap, bu

# Part 3 Write-up: the failure, its cause, and the mitigation

**Failure.** The second-nail-tech question's answer never came back for the owner's own wording. The assistant only got it right by searching twice.

**Cause.** The answer is one item in a seven-feature roadmap paragraph, ranked 7th of 42. The doc's own words barely helped (6th), but the question was nearly twice as close to the item alone (0.499) as to the whole paragraph (0.269). Paragraph chunking diluted it. HyDE didn't help either: it invented a "Staff Management" settings screen.

**Fix.** Splitting list paragraphs into one chunk per item moved the answer to 1st. Found@3 went from 0.95 to 1.00, no other question changed, and no judge score went down in the full rerun. The question now takes one search.

**What's left.** The assistant's first search query also changed between runs, so the single search doesn't prove the fix on its own; the owner's question moving from 7th to 1st does. A v2 prompt that searches with the owner's wording first is the next step.